# Calibración del modelo Markov-modulado a opciones de SPY

Calibración reproducible de $(\sigma_0,\sigma_1,\lambda_0,\lambda_1)$ con datos congelados de opciones de SPY.

## Configuración reproducible

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
DATA = PROJECT_ROOT / "data"
OUTPUT = PROJECT_ROOT / "output"

for directory in (DATA, OUTPUT):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {PROJECT_ROOT}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | SciPy: {scipy.__version__}")
print(f"Semilla: {SEED}")


## El problema de calibración

**Modelo.** La volatilidad de $S_t$ es modulada por una cadena de Markov $\varepsilon(t)\in\{0,1\}$ con generador $Q=\left(\begin{smallmatrix}-\lambda_0&\lambda_0\\\lambda_1&-\lambda_1\end{smallmatrix}\right)$ y ley estacionaria $\pi_0=\lambda_1/(\lambda_0+\lambda_1)$, $\pi_1=1-\pi_0$. Bajo la medida martingala $\mathbb{Q}$ con $r_0=r_1=r$,

$$dS_t = S_t\,r\,dt + S_t\,\sigma_{\varepsilon(t)}\,dW_t^{\mathbb{Q}}.$$

**Problema inverso.** Dados precios de mercado $\{C_j^{\text{mkt}}\}$ con strikes $K_j$ y vencimientos $T_j$, se estima $\theta=(\sigma_0,\sigma_1,\lambda_0,\lambda_1)$ minimizando el error en volatilidad implícita, imponiendo $0<\sigma_0<\sigma_1$ (rompe la simetría de etiqueta) y $\lambda_i>0$. El desarrollo sigue las partes del parcial: **Parte 0** (datos), **Parte A** (fórmula analítica), **Parte B** (validación cruzada), **Parte C** (calibración y diagnóstico) y **Parte D** (análisis e interpretación).

## Parte 0 — Datos: construcción del conjunto

El snapshot se descarga una vez y luego se reutilizan `data/spy_options_raw.csv` y
`data/spy_snapshot_meta.json`. Convenciones: ACT/365, tasa plana de `^IRX`,
dividendo anual de SPY, $F=S_0e^{(r-q)T}$ y precio `mid`.

In [ ]:
from scipy.stats import norm
from scipy.optimize import brentq


def bs_price(S0, K, T, r, q, sigma, option_type="call"):
    """Precio Black-Scholes con dividendo continuo q (forma forward)."""
    if T <= 0:
        payoff = max(S0 - K, 0.0) if option_type == "call" else max(K - S0, 0.0)
        return float(payoff)
    F = S0 * np.exp((r - q) * T)
    disc = np.exp(-r * T)
    if sigma <= 0:
        fwd_payoff = max(F - K, 0.0) if option_type == "call" else max(K - F, 0.0)
        return float(disc * fwd_payoff)
    vol = sigma * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * vol * vol) / vol
    d2 = d1 - vol
    if option_type == "call":
        return float(disc * (F * norm.cdf(d1) - K * norm.cdf(d2)))
    return float(disc * (K * norm.cdf(-d2) - F * norm.cdf(-d1)))


def bs_vega(S0, K, T, r, q, sigma):
    """Vega Black-Scholes (derivada del precio respecto a sigma)."""
    if T <= 0 or sigma <= 0:
        return 0.0
    F = S0 * np.exp((r - q) * T)
    disc = np.exp(-r * T)
    vol = sigma * np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * vol * vol) / vol
    return float(disc * F * norm.pdf(d1) * np.sqrt(T))


def implied_vol(price, S0, K, T, r, q, option_type="call",
                lo=1e-4, hi=5.0):
    """IV por Brent. Devuelve NaN si el precio viola cotas de no arbitraje."""
    if not np.isfinite(price) or price <= 0 or T <= 0:
        return np.nan
    disc = np.exp(-r * T)
    fwd = S0 * np.exp((r - q) * T)
    if option_type == "call":
        lower, upper = disc * max(fwd - K, 0.0), disc * fwd
    else:
        lower, upper = disc * max(K - fwd, 0.0), disc * K
    eps = 1e-10
    if price <= lower + eps or price >= upper - eps:
        return np.nan
    f = lambda s: bs_price(S0, K, T, r, q, s, option_type) - price
    try:
        return float(brentq(f, lo, hi, xtol=1e-8, maxiter=200))
    except (ValueError, RuntimeError):
        return np.nan


print("Helpers BS/IV listos:", bs_price(100, 100, 0.5, 0.03, 0.0, 0.2),
      round(implied_vol(bs_price(100, 100, 0.5, 0.03, 0.0, 0.2),
                        100, 100, 0.5, 0.03, 0.0), 4))

In [ ]:
import json
import datetime as dt

TICKER = "SPY"
TARGET_DTE = [30, 60, 90, 120, 180, 270, 360]
DAYCOUNT = 365.0

MONEYNESS_LO, MONEYNESS_HI = 0.80, 1.20
MAX_REL_SPREAD = 0.50
MIN_VOLUME = 1
MIN_OPEN_INTEREST = 10
MIN_MID = 0.05
IV_LO, IV_HI = 0.01, 3.0

RAW_CSV = DATA / "spy_options_raw.csv"
FILTERED_CSV = DATA / "spy_options_filtered.csv"
META_JSON = DATA / "spy_snapshot_meta.json"

FORCE_DOWNLOAD = False

print("Config de datos lista. Snapshot crudo:", RAW_CSV.name)

In [ ]:
import yfinance as yf


def select_expirations(available_expiries, snapshot_date, target_days):
    """Para cada DTE objetivo elige el vencimiento disponible mas cercano."""
    days_to_expiry = {
        expiry: (dt.date.fromisoformat(expiry) - snapshot_date).days
        for expiry in available_expiries
    }
    selected = []
    for target in target_days:
        expiry = min(available_expiries, key=lambda value: abs(days_to_expiry[value] - target))
        if expiry not in selected and days_to_expiry[expiry] > 0:
            selected.append(expiry)
    return sorted(selected, key=days_to_expiry.get)


def download_snapshot():
    ticker = yf.Ticker(TICKER)
    snapshot_time = dt.datetime.now(dt.timezone.utc)
    snapshot_date = snapshot_time.date()
    spot = float(ticker.fast_info["last_price"])

    irx_close = yf.Ticker("^IRX").history(period="5d")["Close"].dropna()
    risk_free_rate = float(irx_close.iloc[-1]) / 100.0

    dividends = ticker.dividends.copy()
    dividends.index = pd.to_datetime(dividends.index)
    trailing_dividends = float(
        dividends[dividends.index > dividends.index.max() - pd.Timedelta(days=365)].sum()
    )
    dividend_yield = trailing_dividends / spot

    expiries = select_expirations(list(ticker.options), snapshot_date, TARGET_DTE)
    option_tables = []
    for expiry in expiries:
        chain = ticker.option_chain(expiry)
        for option_type, table in (("call", chain.calls), ("put", chain.puts)):
            options = table.copy()
            options["expiry"] = expiry
            options["type"] = option_type
            option_tables.append(options)
    raw_options = pd.concat(option_tables, ignore_index=True)
    raw_options["snapshot_utc"] = snapshot_time.isoformat()
    raw_options["spot"] = spot
    raw_options.to_csv(RAW_CSV, index=False)

    snapshot_meta = {
        "ticker": TICKER,
        "snapshot_utc": snapshot_time.isoformat(),
        "snapshot_date": snapshot_date.isoformat(),
        "timezone": "UTC",
        "spot": spot,
        "r_flat": risk_free_rate,
        "r_source": "^IRX 13-week T-bill, ultimo cierre",
        "q_flat": dividend_yield,
        "q_source": "dividendos SPY ultimos 365 dias / spot",
        "expiries": expiries,
        "daycount": DAYCOUNT,
        "source": "yfinance",
    }
    META_JSON.write_text(json.dumps(snapshot_meta, indent=2))
    return raw_options, snapshot_meta


if RAW_CSV.exists() and META_JSON.exists() and not FORCE_DOWNLOAD:
    print("Snapshot congelado encontrado; se omite la descarga en vivo.")
else:
    downloaded_options, downloaded_meta = download_snapshot()
    print("Snapshot descargado y congelado.")
    print("  filas crudas:", len(downloaded_options), "| vencimientos:", downloaded_meta["expiries"])

In [ ]:
snapshot_meta = json.loads(META_JSON.read_text())
raw_options = pd.read_csv(RAW_CSV)

S0 = float(snapshot_meta["spot"])
R_FLAT = float(snapshot_meta["r_flat"])
Q_FLAT = float(snapshot_meta["q_flat"])
SNAP_DATE = dt.date.fromisoformat(snapshot_meta["snapshot_date"])

print(f"Snapshot {snapshot_meta['snapshot_utc']} ({snapshot_meta['timezone']})")
print(f"Spot S0 = {S0:.2f} | r = {R_FLAT*100:.3f}% ({snapshot_meta['r_source']})")
print(f"q = {Q_FLAT*100:.3f}% ({snapshot_meta['q_source']})")
print(f"Filas crudas: {len(raw_options)} | vencimientos: {len(snapshot_meta['expiries'])}")
raw_options[["expiry", "type", "strike", "bid", "ask", "volume",
     "openInterest"]].head()

In [ ]:
quotes = raw_options.rename(columns={
    "strike": "K",
    "openInterest": "open_interest",
}).copy()

quotes["snapshot_date"] = snapshot_meta["snapshot_date"]
quotes["expiry_date"] = pd.to_datetime(quotes["expiry"]).dt.date
quotes["dte"] = quotes["expiry_date"].apply(lambda date: (date - SNAP_DATE).days)
quotes["T"] = quotes["dte"] / DAYCOUNT
quotes["S0"] = S0
quotes["r"] = R_FLAT
quotes["q"] = Q_FLAT
quotes["forward"] = quotes["S0"] * np.exp((quotes["r"] - quotes["q"]) * quotes["T"])

quotes["mid"] = (quotes["bid"] + quotes["ask"]) / 2.0
quotes["spread"] = quotes["ask"] - quotes["bid"]
quotes["rel_spread"] = quotes["spread"] / quotes["mid"].replace(0, np.nan)
quotes["moneyness"] = quotes["K"] / quotes["S0"]

print("Columnas construidas. Filas:", len(quotes))
quotes[["expiry", "type", "K", "T", "bid", "ask", "mid", "spread",
    "forward", "moneyness"]].head()

In [ ]:
def apply_filter(frame, mask, name):
    kept = int(mask.sum())
    print(f"  {name:<34s} conserva {kept:5d} / {len(frame):5d}")
    return frame[mask]


print("Filtrado de cotizaciones:")
filtered_quotes = quotes.copy()
filtered_quotes = apply_filter(
    filtered_quotes,
    (filtered_quotes["bid"] > 0)
    & (filtered_quotes["ask"] > 0)
    & (filtered_quotes["ask"] >= filtered_quotes["bid"]),
    "bid/ask validos",
)
filtered_quotes = apply_filter(filtered_quotes, filtered_quotes["mid"] >= MIN_MID, f"mid >= {MIN_MID}")
filtered_quotes = apply_filter(
    filtered_quotes,
    filtered_quotes["rel_spread"] <= MAX_REL_SPREAD,
    f"spread relativo <= {MAX_REL_SPREAD}",
)
filtered_quotes = apply_filter(
    filtered_quotes,
    filtered_quotes["volume"].fillna(0) >= MIN_VOLUME,
    f"volumen >= {MIN_VOLUME}",
)
filtered_quotes = apply_filter(
    filtered_quotes,
    filtered_quotes["open_interest"].fillna(0) >= MIN_OPEN_INTEREST,
    f"open interest >= {MIN_OPEN_INTEREST}",
)
filtered_quotes = apply_filter(
    filtered_quotes,
    filtered_quotes["moneyness"].between(MONEYNESS_LO, MONEYNESS_HI),
    f"{MONEYNESS_LO} <= K/S0 <= {MONEYNESS_HI}",
)

discount = np.exp(-filtered_quotes["r"] * filtered_quotes["T"])
forward = filtered_quotes["forward"]
lower_bound = np.where(
    filtered_quotes["type"] == "call",
    discount * np.maximum(forward - filtered_quotes["K"], 0.0),
    discount * np.maximum(filtered_quotes["K"] - forward, 0.0),
)
upper_bound = np.where(
    filtered_quotes["type"] == "call", discount * forward, discount * filtered_quotes["K"]
)
filtered_quotes = apply_filter(
    filtered_quotes,
    (filtered_quotes["mid"] > lower_bound) & (filtered_quotes["mid"] < upper_bound),
    "cotas de no arbitraje",
)

filtered = filtered_quotes.reset_index(drop=True)
print("Filas tras filtros de mercado:", len(filtered))

In [ ]:
filtered["iv_mkt"] = [
    implied_vol(row.mid, row.S0, row.K, row.T, row.r, row.q, row.type)
    for row in filtered.itertuples()
]
valid_iv = filtered["iv_mkt"].between(IV_LO, IV_HI) & filtered["iv_mkt"].notna()
print(f"  IV finita en rango [{IV_LO}, {IV_HI}]: {int(valid_iv.sum())} / {len(filtered)}")
filtered = filtered[valid_iv].reset_index(drop=True)

filtered["vega"] = [
    bs_vega(row.S0, row.K, row.T, row.r, row.q, row.iv_mkt)
    for row in filtered.itertuples()
]
filtered = filtered[filtered["vega"] > 1e-6].reset_index(drop=True)
print("Filas con IV y vega finitas:", len(filtered))

In [ ]:
MARKET_COLUMNS = ["snapshot_date", "expiry", "type", "S0", "K", "T", "dte",
        "bid", "ask", "mid", "spread", "rel_spread", "volume",
        "open_interest", "r", "q", "forward", "iv_mkt", "vega", "moneyness"]
calib = filtered[MARKET_COLUMNS].sort_values(["expiry", "type", "K"]).reset_index(drop=True)
calib.to_csv(FILTERED_CSV, index=False)
print(f"Conjunto calibrable guardado en {FILTERED_CSV.name}: {len(calib)} filas")
calib.head()

In [ ]:
quote_counts = calib.groupby(["expiry", "type"]).size().unstack(fill_value=0)
quote_counts["total"] = quote_counts.sum(axis=1)
quote_counts.loc["TOTAL"] = quote_counts.sum()
dte_map = calib.groupby("expiry")["dte"].first()
quote_counts.insert(0, "dte", [dte_map.get(expiry, "") for expiry in quote_counts.index])
print("Cotizaciones por vencimiento y tipo:")
quote_counts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for expiry, expiry_quotes in calib.groupby("expiry"):
    calls = expiry_quotes[expiry_quotes["type"] == "call"].sort_values("K")
    axes[0].plot(calls["moneyness"], calls["mid"], marker=".", ms=4, label=expiry)
    axes[1].plot(calls["moneyness"], calls["rel_spread"], marker=".", ms=4, label=expiry)
    axes[2].plot(calls["moneyness"], calls["iv_mkt"], marker=".", ms=4, label=expiry)
axes[0].set(title="Precio mid (calls)", xlabel="K/S0", ylabel="mid")
axes[1].set(title="Spread relativo (calls)", xlabel="K/S0", ylabel="(ask-bid)/mid")
axes[2].set(title="IV de mercado (calls)", xlabel="K/S0", ylabel="IV")
axes[2].legend(fontsize=7, ncol=2)
fig.suptitle(f"SPY snapshot {snapshot_meta['snapshot_date']}  |  S0={S0:.2f}")
fig.tight_layout()
fig.savefig(OUTPUT / "spy_exploratorio.png", dpi=120, bbox_inches="tight")
plt.show()

## Parte A — La fórmula analítica

El precio condicionado al régimen inicial combina la masa sin transición y la
densidad continua de la varianza integrada:

$$
C_i(0,s)=e^{-\lambda_i T}\,\mathrm{BS}\!\left(s,K,T,r(a_i^*);a_i^*\right)
+\int_{a_-}^{a_+}\mathrm{BS}\!\left(s,K,T,r(a);a\right)\,g_i^A(a,T)\,da,
$$

Aquí $a_-=\sigma_0^2T$, $a_+=\sigma_1^2T$ y
$R(a)=r_0T+(r_1-r_0)(a-a_-)/(\sigma_1^2-\sigma_0^2)$. La sustitución
$a=c-h\cos t$ estabiliza la cuadratura; si $\sigma_0\approx\sigma_1$ se usa
directamente Black-Scholes plano.

In [ ]:
import warnings
from scipy.integrate import IntegrationWarning, quad
from scipy.special import ive


def _integrate_safely(function, lower, upper, **options):
    """Integra y conserva advertencias relevantes de QUADPACK."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", IntegrationWarning)
        value, error = quad(function, lower, upper, **options)
    for warning in caught:
        if issubclass(warning.category, IntegrationWarning) and error > 1e-6:
            warnings.warn_explicit(
                warning.message, warning.category, warning.filename, warning.lineno
            )
    return value, error


def bs_total_variance(spot, strike, T, integrated_rate, total_variance, option_type="call"):
    """Black-Scholes parametrizado por tasa integrada y varianza total."""
    spot = float(spot)
    strike = float(strike)
    total_variance = np.asarray(total_variance, dtype=float)
    integrated_rate = np.asarray(integrated_rate, dtype=float)
    output_shape = np.broadcast(total_variance, integrated_rate).shape
    variance = np.broadcast_to(total_variance, output_shape).astype(float)
    rate = np.broadcast_to(integrated_rate, output_shape).astype(float)
    prices = np.empty(output_shape, dtype=float)
    positive_variance = variance > 0.0
    if np.any(~positive_variance):
        forward = spot * np.exp(rate[~positive_variance])
        discount = np.exp(-rate[~positive_variance])
        if option_type == "call":
            prices[~positive_variance] = discount * np.maximum(forward - strike, 0.0)
        else:
            prices[~positive_variance] = discount * np.maximum(strike - forward, 0.0)
    if np.any(positive_variance):
        current_variance = variance[positive_variance]
        current_rate = rate[positive_variance]
        std_dev = np.sqrt(current_variance)
        d1 = (np.log(spot / strike) + current_rate + 0.5 * current_variance) / std_dev
        d2 = d1 - std_dev
        discount = np.exp(-current_rate)
        if option_type == "call":
            prices[positive_variance] = spot * norm.cdf(d1) - strike * discount * norm.cdf(d2)
        else:
            prices[positive_variance] = strike * discount * norm.cdf(-d2) - spot * norm.cdf(-d1)
    return prices if prices.shape else float(prices)


def _integrated_rate(total_variance, T, r0, r1, sigma0, sigma1):
    min_variance = sigma0 ** 2 * T
    variance_gap = sigma1 ** 2 - sigma0 ** 2
    return r0 * T + (r1 - r0) * (total_variance - min_variance) / variance_gap


def _integrated_variance_density(total_variance, T, sigma0, sigma1, lam0, lam1, regime):
    """Densidad continua de la varianza integrada."""
    total_variance = np.asarray(total_variance, dtype=float)
    min_variance = sigma0 ** 2 * T
    max_variance = sigma1 ** 2 * T
    variance_gap = sigma1 ** 2 - sigma0 ** 2
    right = max_variance - total_variance
    left = total_variance - min_variance
    right = np.maximum(right, 0.0)
    left = np.maximum(left, 0.0)
    eta = (2.0 / variance_gap) * np.sqrt(lam0 * lam1 * right * left)
    gamma = (lam0 * right + lam1 * left) / variance_gap
    scale = np.exp(-gamma + eta) / variance_gap
    bessel0 = ive(0, eta)
    bessel1 = ive(1, eta)
    if regime == 0:
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(left > 0, right / left, np.inf)
        transition_term = np.sqrt(lam0 * lam1 * ratio) * bessel1
        return scale * (lam0 * bessel0 + transition_term)
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(right > 0, left / right, np.inf)
    transition_term = np.sqrt(lam0 * lam1 * ratio) * bessel1
    return scale * (lam1 * bessel0 + transition_term)


def _continuous_mass(T, sigma0, sigma1, lam0, lam1, regime):
    min_variance = sigma0 ** 2 * T
    max_variance = sigma1 ** 2 * T
    midpoint = 0.5 * (min_variance + max_variance)
    half_width = 0.5 * (max_variance - min_variance)

    def integrand(t):
        total_variance = midpoint - half_width * np.cos(t)
        density = _integrated_variance_density(
            total_variance, T, sigma0, sigma1, lam0, lam1, regime
        )
        return float(density * half_width * np.sin(t))

    return _integrate_safely(
        integrand, 0.0, np.pi, limit=200, epsabs=1e-12, epsrel=1e-12
    )


def price_analytic_markov(S0, K, T, r0, r1, sigma0, sigma1, lam0, lam1,
                          regime=0, option_type="call",
                          quad_limit=200, degen_tol=1e-10):
    """Precio europeo por la formula de mezcla (Teorema 1). Vale para r0 != r1.

    Atomo de borde e^{-lambda_i T} * BS(.; a_i^*) mas integral de BS(.; a) contra
    la densidad continua g_i^A(a,T). Rama degenerada explicita cuando
    sigma1 -> sigma0 (Delta_sigma pequeno). Devuelve un float."""
    if regime not in (0, 1):
        raise ValueError("regime debe ser 0 o 1.")
    if option_type not in ("call", "put"):
        raise ValueError("option_type debe ser 'call' o 'put'.")
    if S0 <= 0 or K <= 0:
        raise ValueError("S0 y K deben ser positivos.")
    if sigma0 <= 0 or sigma1 < sigma0:
        raise ValueError("se requiere 0 < sigma0 <= sigma1.")
    if lam0 <= 0 or lam1 <= 0:
        raise ValueError("lam0 y lam1 deben ser positivos.")
    if T <= 0:
        payoff = (max(S0 - K, 0.0) if option_type == "call"
                  else max(K - S0, 0.0))
        return float(payoff)

    min_variance = sigma0 ** 2 * T
    max_variance = sigma1 ** 2 * T
    variance_gap = sigma1 ** 2 - sigma0 ** 2

    if variance_gap <= degen_tol * max(sigma1 ** 2, 1e-12):
        if not np.isclose(r0, r1, rtol=1e-10, atol=1e-12):
            raise ValueError(
                "Con sigma0≈sigma1 y r0!=r1, A no identifica los tiempos de "
                "ocupacion; la rama degenerada del Teorema 1 no es aplicable."
            )
        total_variance = 0.5 * (min_variance + max_variance)
        return float(bs_total_variance(S0, K, T, r0 * T, total_variance, option_type))

    if regime == 0:
        boundary_variance = min_variance
        boundary_rate = r0 * T
        atom_mass = np.exp(-lam0 * T)
    else:
        boundary_variance = max_variance
        boundary_rate = r1 * T
        atom_mass = np.exp(-lam1 * T)
    atom_price = atom_mass * float(
        bs_total_variance(S0, K, T, boundary_rate, boundary_variance, option_type)
    )

    midpoint = 0.5 * (min_variance + max_variance)
    half_width = 0.5 * (max_variance - min_variance)

    def integrand(t):
        total_variance = midpoint - half_width * np.cos(t)
        rate = _integrated_rate(total_variance, T, r0, r1, sigma0, sigma1)
        conditional_price = float(
            bs_total_variance(S0, K, T, rate, total_variance, option_type)
        )
        density = float(
            _integrated_variance_density(
                total_variance, T, sigma0, sigma1, lam0, lam1, regime
            )
        )
        return conditional_price * density * half_width * np.sin(t)

    continuous_price, _ = _integrate_safely(
        integrand, 0.0, np.pi, limit=quad_limit, epsabs=1e-11, epsrel=1e-10
    )
    return float(atom_price + continuous_price)


_pc0 = price_analytic_markov(100.0, 100.0, 0.5, 0.03, 0.03, 0.15, 0.40,
                             2.0, 5.0, regime=0, option_type="call")
_pc1 = price_analytic_markov(100.0, 100.0, 0.5, 0.03, 0.03, 0.15, 0.40,
                             2.0, 5.0, regime=1, option_type="call")
print("price_analytic_markov listo.")
print(f"  Caso base K=100 call: C0={_pc0:.6f}  C1={_pc1:.6f}")


### Pruebas internas

Masa total, límite Black-Scholes, cotas, monotonía y paridad call-put.

In [ ]:
print("=== Prueba 1: masa total (objetivo error <= 1e-6) ===")
_param_sets = [
    dict(T=0.5, sigma0=0.15, sigma1=0.40, lam0=2.0, lam1=5.0),
    dict(T=1.0, sigma0=0.10, sigma1=0.60, lam0=0.5, lam1=3.0),
    dict(T=0.25, sigma0=0.20, sigma1=0.30, lam0=4.0, lam1=1.0),
]
_max_mass_err = 0.0
for ps in _param_sets:
    for reg in (0, 1):
        atom = np.exp(-(ps["lam0"] if reg == 0 else ps["lam1"]) * ps["T"])
        cont, qerr = _continuous_mass(ps["T"], ps["sigma0"], ps["sigma1"],
                                      ps["lam0"], ps["lam1"], reg)
        total = atom + cont
        err = abs(total - 1.0)
        _max_mass_err = max(_max_mass_err, err)
        print(f"  T={ps['T']:.2f} s0={ps['sigma0']:.2f} s1={ps['sigma1']:.2f} "
              f"l0={ps['lam0']:.1f} l1={ps['lam1']:.1f} reg={reg} | "
              f"atomo={atom:.6f} integral={cont:.6f} total={total:.10f} "
              f"err={err:.2e}")
print(f"  -> max error de masa = {_max_mass_err:.3e}  "
      f"({'OK' if _max_mass_err <= 1e-6 else 'FALLA'} <= 1e-6)")


In [ ]:
print("=== Prueba 2: caso singular sigma0 -> sigma1 (BS plano) ===")
_S0, _K, _T, _r = 100.0, 100.0, 0.5, 0.03
_sig = 0.25
_bs_flat = float(bs_total_variance(_S0, _K, _T, _r * _T, _sig ** 2 * _T, "call"))
print(f"  BS plano (sigma={_sig}) call K={_K}: {_bs_flat:.8f}")
for _eps in (1e-2, 1e-4, 1e-6, 1e-9):
    _s0 = _sig - 0.5 * _eps
    _s1 = _sig + 0.5 * _eps
    _c = price_analytic_markov(_S0, _K, _T, _r, _r, _s0, _s1, 2.0, 5.0,
                               regime=0, option_type="call")
    print(f"  sigma1-sigma0={_eps:.0e}  C0={_c:.8f}  |C0-BSplano|={abs(_c-_bs_flat):.2e}")
_c_exact = price_analytic_markov(_S0, _K, _T, _r, _r, _sig, _sig, 2.0, 5.0,
                                 regime=0, option_type="call")
print(f"  sigma0==sigma1 (rama explicita): C0={_c_exact:.8f}  "
      f"|dif|={abs(_c_exact-_bs_flat):.2e}")
try:
    price_analytic_markov(_S0, _K, _T, 0.02, 0.06, _sig, _sig, 2.0, 5.0)
    _degen_unequal_ok = False
except ValueError:
    _degen_unequal_ok = True
print("  sigma0==sigma1 con r0!=r1 se rechaza explicitamente:", _degen_unequal_ok)


In [ ]:
print("=== Prueba 3: cota BS(a-) <= C_i <= BS(a+) (r0=r1) ===")
_bateria = [
    dict(S0=100, K=90,  T=0.5, r=0.03, s0=0.15, s1=0.40, l0=2.0, l1=5.0),
    dict(S0=100, K=100, T=0.5, r=0.03, s0=0.15, s1=0.40, l0=2.0, l1=5.0),
    dict(S0=100, K=110, T=0.5, r=0.03, s0=0.15, s1=0.40, l0=2.0, l1=5.0),
    dict(S0=100, K=100, T=1.0, r=0.05, s0=0.10, s1=0.50, l0=1.0, l1=1.0),
    dict(S0=120, K=100, T=0.25,r=0.01, s0=0.20, s1=0.35, l0=3.0, l1=0.5),
    dict(S0=80,  K=100, T=2.0, r=0.04, s0=0.12, s1=0.45, l0=0.3, l1=4.0),
]
_all_ok = True
for _opt in ("call", "put"):
    for p in _bateria:
        am = p["s0"] ** 2 * p["T"]
        ap = p["s1"] ** 2 * p["T"]
        R = p["r"] * p["T"]
        bs_lo = float(bs_total_variance(p["S0"], p["K"], p["T"], R, am, _opt))
        bs_hi = float(bs_total_variance(p["S0"], p["K"], p["T"], R, ap, _opt))
        lo, hi = min(bs_lo, bs_hi), max(bs_lo, bs_hi)
        for reg in (0, 1):
            C = price_analytic_markov(p["S0"], p["K"], p["T"], p["r"], p["r"],
                                      p["s0"], p["s1"], p["l0"], p["l1"],
                                      regime=reg, option_type=_opt)
            ok = (lo - 1e-9) <= C <= (hi + 1e-9)
            _all_ok = _all_ok and ok
            if not ok:
                print(f"  FALLA {_opt} K={p['K']} reg={reg}: {lo:.4f} <= {C:.4f} <= {hi:.4f}")
print(f"  -> cota se cumple en toda la bateria (call y put): {_all_ok}")


In [ ]:
print("=== Prueba 4: sanidad financiera ===")
_S0, _T, _r = 100.0, 0.5, 0.03
_s0, _s1, _l0, _l1 = 0.15, 0.40, 2.0, 5.0
_Ks = np.array([80, 85, 90, 95, 100, 105, 110, 115, 120], dtype=float)

_pos_ok = True
_arb_ok = True
_mono_call_ok = True
_mono_put_ok = True
_max_parity = 0.0
_has_nan = False

for reg in (0, 1):
    calls = np.array([price_analytic_markov(_S0, k, _T, _r, _r, _s0, _s1,
                                            _l0, _l1, reg, "call") for k in _Ks])
    puts = np.array([price_analytic_markov(_S0, k, _T, _r, _r, _s0, _s1,
                                           _l0, _l1, reg, "put") for k in _Ks])
    if not (np.all(np.isfinite(calls)) and np.all(np.isfinite(puts))):
        _has_nan = True
    _pos_ok = _pos_ok and np.all(calls >= -1e-10) and np.all(puts >= -1e-10)
    disc = np.exp(-_r * _T)
    fwd = _S0 * np.exp(_r * _T)
    call_lo = disc * np.maximum(fwd - _Ks, 0.0)
    call_hi = disc * fwd
    put_lo = disc * np.maximum(_Ks - fwd, 0.0)
    put_hi = disc * _Ks
    _arb_ok = (_arb_ok and np.all(calls >= call_lo - 1e-8)
               and np.all(calls <= call_hi + 1e-8)
               and np.all(puts >= put_lo - 1e-8)
               and np.all(puts <= put_hi + 1e-8))
    _mono_call_ok = _mono_call_ok and np.all(np.diff(calls) <= 1e-8)
    _mono_put_ok = _mono_put_ok and np.all(np.diff(puts) >= -1e-8)
    parity = (calls - puts) - (_S0 - _Ks * disc)
    _max_parity = max(_max_parity, float(np.max(np.abs(parity))))

print(f"  sin NaN/inf:            {not _has_nan}")
print(f"  positividad:            {_pos_ok}")
print(f"  no arbitraje:           {_arb_ok}")
print(f"  call decreciente en K:  {_mono_call_ok}")
print(f"  put creciente en K:     {_mono_put_ok}")
print(f"  max|paridad C-P|:       {_max_parity:.3e}  "
      f"({'OK' if _max_parity <= 1e-6 else 'FALLA'} <= 1e-6)")


## Parte B — Validación cruzada de las tres rutas

### Implementación de las rutas B (COS/FFT) y C (EDP)

Se reutilizan la función característica, COS y Carr-Madan del Taller 2, y la
EDP Crank-Nicolson del Taller 3. Durante la calibración se usa $r_0=r_1=r$ y
el dividendo se absorbe en $S_0e^{-qT}$.

#### Función característica

Forma cerrada estable y control independiente mediante exponencial matricial.

In [ ]:
import numpy as np
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from scipy.stats import norm


def char_func_markov(z, T, r0, r1, sigma0, sigma1, lam0, lam1, regime=0):
    """phi_i(z, T) del log-retorno bajo Q (forma cerrada estable)."""
    z = np.asarray(z, dtype=np.complex128)

    muQ0 = r0 - 0.5 * sigma0 ** 2
    muQ1 = r1 - 0.5 * sigma1 ** 2

    muQ_plus = (muQ0 + muQ1) / 2.0
    muQ_minus = (muQ0 - muQ1) / 2.0
    sig_plus = (sigma0 ** 2 + sigma1 ** 2) / 4.0
    sig_minus = (sigma0 ** 2 - sigma1 ** 2) / 4.0
    lam_plus = (lam0 + lam1) / 2.0
    lam_minus = (lam0 - lam1) / 2.0

    rho_plus = 1j * z * muQ_plus - sig_plus * z ** 2 - lam_plus
    rho_minus = 1j * z * muQ_minus - sig_minus * z ** 2 - lam_minus

    D = rho_minus ** 2 + lam0 * lam1
    sqrt_D = np.sqrt(D)
    safe_sqrt_D = np.where(sqrt_D == 0, 1.0, sqrt_D)

    if regime == 0:
        c = (rho_minus + lam0) / safe_sqrt_D
        limit_val = np.exp(T * rho_plus) * (1.0 + (rho_minus + lam0) * T)
    else:
        c = -(rho_minus - lam1) / safe_sqrt_D
        limit_val = np.exp(T * rho_plus) * (1.0 - (rho_minus - lam1) * T)

    term1 = 0.5 * np.exp(T * (rho_plus + sqrt_D)) * (1.0 + c)
    term2 = 0.5 * np.exp(T * (rho_plus - sqrt_D)) * (1.0 - c)
    stable_phi = term1 + term2
    return np.where(sqrt_D == 0, limit_val, stable_phi)


def char_func_markov_matrix(z, T, r0, r1, sigma0, sigma1, lam0, lam1, regime=0):
    """Cross-check de phi_i via exp de la matriz generadora 2x2 (lento)."""
    z = np.atleast_1d(np.asarray(z, dtype=complex))
    muQ0 = r0 - 0.5 * sigma0 ** 2
    muQ1 = r1 - 0.5 * sigma1 ** 2
    e_i = (np.array([1.0, 0.0], dtype=complex) if regime == 0
           else np.array([0.0, 1.0], dtype=complex))
    ones = np.ones(2, dtype=complex)
    out = np.empty_like(z, dtype=complex)
    for j, zz in enumerate(z):
        a0 = 1j * zz * muQ0 - 0.5 * sigma0 ** 2 * zz ** 2 - lam0
        a1 = 1j * zz * muQ1 - 0.5 * sigma1 ** 2 * zz ** 2 - lam1
        A = np.array([[a0, lam0], [lam1, a1]], dtype=complex)
        vals, vecs = np.linalg.eig(A * T)
        expA = vecs @ np.diag(np.exp(vals)) @ np.linalg.inv(vecs)
        out[j] = e_i @ expA @ ones
    return out


def stationary_weights(lam0, lam1):
    """Pesos estacionarios (pi0, pi1) de la cadena de regimenes."""
    tot = lam0 + lam1
    pi0 = lam1 / tot if tot > 0 else 0.5
    return pi0, 1.0 - pi0


_z = np.linspace(-20, 20, 41)
_phi_bs = np.exp(1j * _z * (0.03 - 0.5 * 0.2 ** 2) * 0.5 - 0.5 * 0.2 ** 2 * _z ** 2 * 0.5)
_phi_a = char_func_markov(_z, 0.5, 0.03, 0.03, 0.2, 0.2, 0.0, 0.0, regime=0)
_phi_b = char_func_markov_matrix(_z, 0.5, 0.03, 0.03, 0.15, 0.40, 2.0, 5.0, regime=0)
_phi_c = char_func_markov(_z, 0.5, 0.03, 0.03, 0.15, 0.40, 2.0, 5.0, regime=0)
print("max|phi_markov - phi_BS| (sin regimen):", float(np.max(np.abs(_phi_a - _phi_bs))))
print("max|forma cerrada - via matriz|       :", float(np.max(np.abs(_phi_b - _phi_c))))


#### Black-Scholes vectorizado

In [ ]:
def bs_call_cos(S0, K, T, r, sigma):
    """Call europea Black-Scholes, vectorizable sobre K (tasa constante r)."""
    K = np.asarray(K, dtype=float)
    sqrtTv = sigma * np.sqrt(T)
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / sqrtTv
    d2 = d1 - sqrtTv
    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def bs_put_cos(S0, K, T, r, sigma):
    """Put europea Black-Scholes (paridad), vectorizable sobre K."""
    K = np.asarray(K, dtype=float)
    sqrtTv = sigma * np.sqrt(T)
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / sqrtTv
    d2 = d1 - sqrtTv
    return K * np.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)


def bs_vega_cos(S0, K, T, r, sigma):
    """Vega Black-Scholes (igual para call y put), vectorizable sobre K."""
    K = np.asarray(K, dtype=float)
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    return S0 * norm.pdf(d1) * np.sqrt(T)


_K = np.array([90.0, 100.0, 110.0])
_C = bs_call_cos(100.0, _K, 0.5, 0.03, 0.2)
_P = bs_put_cos(100.0, _K, 0.5, 0.03, 0.2)
print("max|paridad C-P - (S0-K e^{-rT})|:",
      float(np.max(np.abs((_C - _P) - (100.0 - _K * np.exp(-0.03 * 0.5))))))


#### Método COS

In [ ]:
def cos_method(S0, K, T, r, sigma0, sigma1, lam0, lam1,
               regime=0, N=256, L=10, option_type="call"):
    """Valoracion COS de calls/puts europeas Markov-moduladas. Vectorizada en K."""
    K = np.atleast_1d(np.float64(K))
    x = np.log(S0 / K)

    pi0 = lam1 / (lam0 + lam1) if (lam0 + lam1) > 0 else 0.5
    mu_avg = pi0 * (r - 0.5 * sigma0 ** 2) + (1 - pi0) * (r - 0.5 * sigma1 ** 2)
    var_avg = pi0 * sigma0 ** 2 + (1 - pi0) * sigma1 ** 2

    c1 = mu_avg * T
    c2 = var_avg * T
    a = c1 - L * np.sqrt(c2)
    b = c1 + L * np.sqrt(c2)

    k_arr = np.arange(N)
    u = k_arr * np.pi / (b - a)
    phi_vals = char_func_markov(u, T, r, r, sigma0, sigma1, lam0, lam1, regime)

    def chi_k(c, d):
        arg_d = k_arr * np.pi * (d - a) / (b - a)
        arg_c = k_arr * np.pi * (c - a) / (b - a)
        denom = 1.0 + u ** 2
        num = (np.cos(arg_d) * np.exp(d) - np.cos(arg_c) * np.exp(c)
               + u * (np.sin(arg_d) * np.exp(d) - np.sin(arg_c) * np.exp(c)))
        return num / denom

    def psi_k(c, d):
        out = np.zeros_like(k_arr, dtype=float)
        arg_d = k_arr * np.pi * (d - a) / (b - a)
        arg_c = k_arr * np.pi * (c - a) / (b - a)
        nz = k_arr != 0
        out[nz] = (np.sin(arg_d[nz]) - np.sin(arg_c[nz])) * (b - a) / (k_arr[nz] * np.pi)
        out[~nz] = d - c
        return out

    if option_type == "call":
        Vk = 2.0 / (b - a) * (chi_k(0.0, b) - psi_k(0.0, b))
    elif option_type == "put":
        Vk = 2.0 / (b - a) * (-chi_k(a, 0.0) + psi_k(a, 0.0))
    else:
        raise ValueError("option_type debe ser 'call' o 'put'.")

    base = np.exp(-1j * u * a)
    prices = np.zeros(len(K))
    for j, xj in enumerate(x):
        coeffs = np.real(phi_vals * np.exp(1j * u * xj) * base)
        coeffs[0] *= 0.5
        prices[j] = K[j] * np.exp(-r * T) * np.sum(coeffs * Vk)
    return prices


_K = np.array([90.0, 100.0, 110.0])
_cos_bs = cos_method(100.0, _K, 0.5, 0.03, 0.2, 0.2, 0.0, 0.0,
                     regime=0, N=512, option_type="call")
_ref_bs = bs_call_cos(100.0, _K, 0.5, 0.03, 0.2)
print("COS sin regimen vs BS, max abs:", float(np.max(np.abs(_cos_bs - _ref_bs))))
print("COS call (caso base, regimen 0) K=90,100,110:",
      np.round(cos_method(100.0, _K, 0.5, 0.03, 0.15, 0.40, 2.0, 5.0,
                          regime=0, N=512, option_type="call"), 6))


#### Carr-Madan FFT

In [ ]:
def carr_madan_fft(S0, T, r, sigma0, sigma1, lam0, lam1,
                   regime=0, N=4096, alpha=1.5, eta=0.25):
    """Calls europeas por Carr-Madan FFT (Markov-modulado). Devuelve (K, precios)."""
    if N <= 0 or (N & (N - 1)) != 0:
        raise ValueError("N debe ser potencia de 2.")
    if alpha <= 0 or eta <= 0:
        raise ValueError("alpha y eta deben ser positivos.")
    if regime not in (0, 1):
        raise ValueError("regime debe ser 0 o 1.")

    dk = 2.0 * np.pi / (N * eta)
    b = 0.5 * N * dk
    j = np.arange(N)
    nu = j * eta

    w = np.ones(N)            # pesos Simpson 1/3 con correccion Kronecker
    w[0] = 1.0 / 3.0
    w[1::2] = 4.0 / 3.0
    w[2::2] = 2.0 / 3.0

    z = nu - 1j * (alpha + 1.0)
    phi = char_func_markov(z, T, r, r, sigma0, sigma1, lam0, lam1, regime=regime)

    denom = (alpha + 1j * nu) * (alpha + 1.0 + 1j * nu)
    psi = np.exp(-r * T) * phi / denom
    fft_in = np.exp(1j * b * nu) * psi * eta * w
    y = np.fft.fft(fft_in)

    u = -b + j * dk                       # log-moneyness ln(K/S0)
    strikes = S0 * np.exp(u)
    prices = S0 * np.exp(-alpha * u) * np.real(y) / np.pi
    prices = np.maximum(prices, 0.0)

    mask = (strikes > 0.3 * S0) & (strikes < 3.0 * S0)
    return strikes[mask], prices[mask]


def cos_vs_fft_call(S0, K, T, r, sigma0, sigma1, lam0, lam1,
                    regime=0, N_cos=256, N_fft=4096, alpha=1.5, eta=0.25):
    """Cross-check COS vs FFT para calls; interpola FFT a los strikes K."""
    K = np.atleast_1d(np.float64(K))
    C_cos = cos_method(S0, K, T, r, sigma0, sigma1, lam0, lam1,
                       regime=regime, N=N_cos, option_type="call")
    K_grid, C_grid = carr_madan_fft(S0, T, r, sigma0, sigma1, lam0, lam1,
                                    regime=regime, N=N_fft, alpha=alpha, eta=eta)
    C_fft = interp1d(K_grid, C_grid, kind="cubic")(K)
    return {"K": K, "C_cos": C_cos, "C_fft": C_fft,
            "max_abs_diff": float(np.max(np.abs(C_cos - C_fft)))}


_Kg, _Cg = carr_madan_fft(100.0, 0.5, 0.03, 0.2, 0.2, 0.0, 0.0, regime=0)
_sel = (_Kg > 80) & (_Kg < 120)
_fft_err = float(np.max(np.abs(_Cg[_sel] - bs_call_cos(100.0, _Kg[_sel], 0.5, 0.03, 0.2))))
print("FFT sin regimen vs BS (80<K<120), max abs:", _fft_err)


#### Inversión de volatilidad implícita

In [ ]:
def implied_vol_call(C_market, S0, K, T, r, lo=1e-4, hi=5.0):
    """IV BS de una call por brentq, con chequeo de cotas. NaN si fuera de rango."""
    intrinsic = max(S0 - K * np.exp(-r * T), 0.0)
    if (not np.isfinite(C_market)) or C_market <= intrinsic - 1e-12 or C_market >= S0:
        return np.nan
    try:
        return brentq(lambda s: float(bs_call_cos(S0, K, T, r, s)) - C_market,
                      lo, hi, xtol=1e-10, maxiter=200)
    except ValueError:
        return np.nan


def implied_vol_smile(C_market, S0, K, T, r):
    """Vectoriza implied_vol_call sobre arrays de precios y strikes."""
    C_market = np.atleast_1d(C_market)
    K = np.atleast_1d(K)
    return np.array([implied_vol_call(c, S0, k, T, r) for c, k in zip(C_market, K)])


_K = np.array([90.0, 100.0, 110.0])
_iv = implied_vol_smile(bs_call_cos(100.0, _K, 0.5, 0.03, 0.25), 100.0, _K, 0.5, 0.03)
print("IV round-trip (sigma=0.25):", np.round(_iv, 6))


#### EDP acoplada Crank-Nicolson

Diferencias finitas con theta-esquema para la EDP acoplada de dos regimenes
(tiempo hacia atras \(\tau=T-t\)):

\[
\partial_\tau V_i = \tfrac12\sigma_i^2 S^2\,\partial_{SS}V_i
 + r_i S\,\partial_S V_i - r_i V_i + \lambda_i\,(V_{1-i}-V_i),\quad i\in\{0,1\}.
\]

La validación usa $\theta=1/2$ y una malla fina para alcanzar error cercano a $10^{-3}$.

In [ ]:
from scipy.sparse import diags, identity, bmat
from scipy.sparse.linalg import splu


def _build_coupled_operator(M, sigma0, sigma1, r0, r1, lam0, lam1):
    """Ensambla L (2M x 2M) y devuelve c0[M-1], c1[M-1] para ghosts."""
    j = np.arange(M + 1)  # 0, 1, ..., M

    a0 = 0.5 * sigma0 ** 2 * j ** 2 - 0.5 * r0 * j
    b0 = -sigma0 ** 2 * j ** 2 - r0 - lam0
    c0 = 0.5 * sigma0 ** 2 * j ** 2 + 0.5 * r0 * j
    a1 = 0.5 * sigma1 ** 2 * j ** 2 - 0.5 * r1 * j
    b1 = -sigma1 ** 2 * j ** 2 - r1 - lam1
    c1 = 0.5 * sigma1 ** 2 * j ** 2 + 0.5 * r1 * j

    L0 = diags([a0[1:M], b0[:M], c0[:M - 1]], offsets=[-1, 0, 1],
               shape=(M, M), format="csr")
    L1 = diags([a1[1:M], b1[:M], c1[:M - 1]], offsets=[-1, 0, 1],
               shape=(M, M), format="csr")
    Iblock = identity(M, format="csr")
    L = bmat([[L0, lam0 * Iblock], [lam1 * Iblock, L1]], format="csr")
    return L, c0[M - 1], c1[M - 1]


def fd_european_put_markov(S0, K, T, r0, r1, sigma0, sigma1, lam0, lam1,
                           regime=0, Smax=None, M=200, Nt=200, theta=0.5):
    """PUT europea Markov-modulada por theta-esquema. Retorna V_regime(S0, 0)."""
    if Smax is None:
        Smax = 4.0 * K
    dtau = T / Nt
    S = np.linspace(0.0, Smax, M + 1)

    L, _, _ = _build_coupled_operator(M, sigma0, sigma1, r0, r1, lam0, lam1)
    payoff = np.maximum(K - S[:M], 0.0)
    U = np.concatenate([payoff, payoff])

    I2M = identity(2 * M, format="csr")
    A_imp = (I2M - theta * dtau * L).tocsc()
    B_exp = (I2M + (1.0 - theta) * dtau * L).tocsr()
    lu = splu(A_imp)
    for _ in range(Nt):
        U = lu.solve(B_exp @ U)

    V_grid = U[:M] if regime == 0 else U[M:]
    return float(np.interp(S0, S[:M], V_grid))


def fd_european_call_markov(S0, K, T, r0, r1, sigma0, sigma1, lam0, lam1,
                            regime=0, Smax=None, M=200, Nt=200, theta=0.5):
    """CALL europea Markov-modulada (theta-esquema, frontera Smax via ghost)."""
    if Smax is None:
        Smax = 4.0 * max(K, S0)
    dtau = T / Nt
    S = np.linspace(0.0, Smax, M + 1)

    L, cM0, cM1 = _build_coupled_operator(M, sigma0, sigma1, r0, r1, lam0, lam1)
    payoff = np.maximum(S[:M] - K, 0.0)
    U = np.concatenate([payoff, payoff])

    I2M = identity(2 * M, format="csr")
    A_imp = (I2M - theta * dtau * L).tocsc()
    B_exp = (I2M + (1.0 - theta) * dtau * L).tocsr()
    lu = splu(A_imp)
    for n in range(Nt):
        tau_n = n * dtau
        tau_n1 = (n + 1) * dtau
        VM0_n = Smax - K * np.exp(-r0 * tau_n)
        VM1_n = Smax - K * np.exp(-r1 * tau_n)
        VM0_n1 = Smax - K * np.exp(-r0 * tau_n1)
        VM1_n1 = Smax - K * np.exp(-r1 * tau_n1)

        ghost = np.zeros(2 * M)
        ghost[M - 1] += dtau * (1.0 - theta) * cM0 * VM0_n
        ghost[2 * M - 1] += dtau * (1.0 - theta) * cM1 * VM1_n
        ghost[M - 1] += dtau * theta * cM0 * VM0_n1
        ghost[2 * M - 1] += dtau * theta * cM1 * VM1_n1
        U = lu.solve(B_exp @ U + ghost)

    V_grid = U[:M] if regime == 0 else U[M:]
    return float(np.interp(S0, S[:M], V_grid))


_P_bs = bs_put_cos(100.0, 100.0, 0.5, 0.05, 0.20)
_P_fd = fd_european_put_markov(100.0, 100.0, 0.5, 0.05, 0.05, 0.20, 0.20,
                               0.0, 0.0, regime=0, M=400, Nt=400, theta=0.5)
print("PUT EDP degenerada vs BS:", round(_P_fd, 6), "vs", round(_P_bs, 6),
      "| abs:", abs(_P_fd - _P_bs))


### Tabla de validación A/B/C

Se comparan calls por las tres rutas para $K\in\{90,100,110\}$ y ambos
regímenes, con tolerancia objetivo de $10^{-3}$.

In [ ]:
import time as _time
import pandas as pd

COS_N = 1024
COS_L = 12
FFT_N = 2 ** 16
FFT_ALPHA = 1.5
FFT_ETA = 0.05
EDP_M = 800
EDP_NT = 800
EDP_THETA = 0.5

print("Parametros de validacion cruzada:")
print(f"  COS : N={COS_N}, L={COS_L}")
print(f"  FFT : N={FFT_N}, alpha={FFT_ALPHA}, eta={FFT_ETA}")
print(f"  EDP : M={EDP_M}, Nt={EDP_NT}, theta={EDP_THETA}")


In [ ]:
def tabla_validacion_abc(S0, T, r, sigma0, sigma1, lam0, lam1, strikes,
                         cos_N=COS_N, cos_L=COS_L,
                         fft_N=FFT_N, fft_alpha=FFT_ALPHA, fft_eta=FFT_ETA,
                         edp_M=EDP_M, edp_Nt=EDP_NT, edp_theta=EDP_THETA):
    """Devuelve (df, tiempos) con precios call por A, B(COS), B(FFT control), C
    para cada strike y ambos regimenes, mas las diferencias absolutas."""
    strikes = np.asarray(strikes, dtype=float)
    rows = []
    tiempos = {}

    for regime in (0, 1):
        # --- Ruta B: COS vectorizado sobre strikes (un solo r) ---
        t0 = _time.perf_counter()
        C_cos = cos_method(S0, strikes, T, r, sigma0, sigma1, lam0, lam1,
                           regime=regime, N=cos_N, L=cos_L, option_type="call")
        tiempos[("COS", regime)] = _time.perf_counter() - t0

        # --- Ruta B control: FFT Carr-Madan (malla -> interpolar a strikes) ---
        t0 = _time.perf_counter()
        K_grid, C_grid = carr_madan_fft(S0, T, r, sigma0, sigma1, lam0, lam1,
                                        regime=regime, N=fft_N,
                                        alpha=fft_alpha, eta=fft_eta)
        C_fft = interp1d(K_grid, C_grid, kind="cubic")(strikes)
        tiempos[("FFT", regime)] = _time.perf_counter() - t0

        for j, K in enumerate(strikes):
            # --- Ruta A: mezcla analitica (r0=r1=r) ---
            t0 = _time.perf_counter()
            C_a = price_analytic_markov(S0, K, T, r, r, sigma0, sigma1,
                                        lam0, lam1, regime=regime,
                                        option_type="call")
            t_a = _time.perf_counter() - t0

            # --- Ruta C: EDP CN call directa ---
            t0 = _time.perf_counter()
            C_c = fd_european_call_markov(S0, K, T, r, r, sigma0, sigma1,
                                          lam0, lam1, regime=regime,
                                          M=edp_M, Nt=edp_Nt, theta=edp_theta)
            t_c = _time.perf_counter() - t0

            rows.append({
                "regime": regime, "K": K,
                "A": C_a, "B_COS": float(C_cos[j]),
                "B_FFT": float(C_fft[j]), "C_EDP": C_c,
                "|A-B|": abs(C_a - float(C_cos[j])),
                "|A-C|": abs(C_a - C_c),
                "|B-C|": abs(float(C_cos[j]) - C_c),
                "|COS-FFT|": abs(float(C_cos[j]) - float(C_fft[j])),
                "t_A": t_a, "t_C": t_c,
            })
            tiempos.setdefault(("A", regime), 0.0)
            tiempos[("A", regime)] += t_a
            tiempos.setdefault(("EDP", regime), 0.0)
            tiempos[("EDP", regime)] += t_c

    df = pd.DataFrame(rows)
    return df, tiempos


print("tabla_validacion_abc lista.")


In [ ]:
S0_b, T_b, r_b = 100.0, 0.5, 0.03
s0_b, s1_b = 0.15, 0.40
l0_b, l1_b = 2.0, 5.0
K_b = [90.0, 100.0, 110.0]

df_base, tiempos_base = tabla_validacion_abc(S0_b, T_b, r_b, s0_b, s1_b,
                                             l0_b, l1_b, K_b)

pd.set_option("display.float_format", lambda v: f"{v:.6f}")
print("=== CASO BASE: S0=100, T=0.5, r=0.03, sig0=0.15, sig1=0.40, "
      "lam0=2, lam1=5 ===\n")
print("Precios CALL por ruta (3 strikes x 2 regimenes):")
print(df_base[["regime", "K", "A", "B_COS", "B_FFT", "C_EDP"]].to_string(index=False))

print("\nDiferencias absolutas entre rutas:")
diff_cols = ["regime", "K", "|A-B|", "|A-C|", "|B-C|", "|COS-FFT|"]
with pd.option_context("display.float_format", lambda v: f"{v:.2e}"):
    print(df_base[diff_cols].to_string(index=False))

max_AB = df_base["|A-B|"].max()
max_AC = df_base["|A-C|"].max()
max_BC = df_base["|B-C|"].max()
print(f"\nmax|A-B| = {max_AB:.3e}   max|A-C| = {max_AC:.3e}   "
      f"max|B-C| = {max_BC:.3e}")
TOL_ABC = 1e-3
ok_base = max(max_AB, max_AC, max_BC) <= TOL_ABC
print(f"Todas las diferencias <= {TOL_ABC:.0e}:  {ok_base}")


### Diagnóstico numérico

La Ruta A está limitada por cuadratura, COS/FFT por truncamiento espectral y la
EDP por discretización. Esta última domina las diferencias; por costo y precisión,
COS se usa dentro de la calibración.

## Parte C — Calibración a datos reales

### Precio del modelo, objetivo y ponderación

Se usa mezcla estacionaria, spot ajustado por dividendos y COS agrupado por
vencimiento. Una transformación logística acotada garantiza
$0<\sigma_0<\sigma_1$ y $\lambda_i>0$. El objetivo es RMSE en IV con pesos
uniformes o por vega.

In [ ]:

SIGMA0_CEIL = 0.60
SIGMA1_CEIL = 0.90
LAM_FLOOR = 0.05
LAM_CEIL = 20.0


def _sigmoid(x):
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(x, dtype=float)))


def _logit(p):
    p = float(np.clip(p, 1e-12, 1.0 - 1e-12))
    return float(np.log(p / (1.0 - p)))


def theta_from_u(u):
    """u in R^4 -> theta=(sigma0, sigma1, lam0, lam1) factible y acotada."""
    u = np.asarray(u, dtype=float)
    sigma0 = SIGMA0_CEIL * _sigmoid(u[0])
    sigma1 = sigma0 + (SIGMA1_CEIL - sigma0) * _sigmoid(u[1])
    lam0 = LAM_FLOOR + (LAM_CEIL - LAM_FLOOR) * _sigmoid(u[2])
    lam1 = LAM_FLOOR + (LAM_CEIL - LAM_FLOOR) * _sigmoid(u[3])
    return float(sigma0), float(sigma1), float(lam0), float(lam1)


def u_from_theta(sigma0, sigma1, lam0, lam1):
    """theta factible -> u in R^4 (inversa exacta de theta_from_u)."""
    u0 = _logit(sigma0 / SIGMA0_CEIL)
    u1 = _logit((sigma1 - sigma0) / (SIGMA1_CEIL - sigma0))
    u2 = _logit((lam0 - LAM_FLOOR) / (LAM_CEIL - LAM_FLOOR))
    u3 = _logit((lam1 - LAM_FLOOR) / (LAM_CEIL - LAM_FLOOR))
    return np.array([u0, u1, u2, u3], dtype=float)


_theta_ref = (0.15, 0.40, 2.0, 5.0)
_u_ref = u_from_theta(*_theta_ref)
_theta_rt = theta_from_u(_u_ref)
print("Round-trip theta -> u -> theta (transform acotado):")
print("  theta original :", tuple(round(v, 6) for v in _theta_ref))
print("  theta reconstr.:", tuple(round(v, 6) for v in _theta_rt))
print("  max abs error  :", float(np.max(np.abs(np.array(_theta_ref) - np.array(_theta_rt)))))
print(f"  techos: sigma0<{SIGMA0_CEIL}, sigma1<{SIGMA1_CEIL}")
print(f"  intensidades: {LAM_FLOOR}<lambda_i<{LAM_CEIL}")
print("  sigma1(u1=+inf) ->", round(theta_from_u([0.0, 1e3, 0.0, 0.0])[1], 6), "(<= techo)")
print("  factible (0<s0<s1, lam>0):",
      0 < _theta_rt[0] < _theta_rt[1] and _theta_rt[2] > 0 and _theta_rt[3] > 0)


In [ ]:


_p0, _p1 = stationary_weights(2.0, 5.0)
print(f"Caso base lam0=2, lam1=5 -> pi0={_p0:.6f}, pi1={_p1:.6f}, suma={_p0 + _p1:.12f}")
_q0, _q1 = stationary_weights(8.0, 5.0)
print(f"Subiendo lam0 a 8     -> pi0={_q0:.6f}, pi1={_q1:.6f}, suma={_q0 + _q1:.12f}")
print(f"  pi1 sube con lam0 (regimen 0 menos persistente): {_q1 > _p1}")
assert abs((_p0 + _p1) - 1.0) < 1e-12 and abs((_q0 + _q1) - 1.0) < 1e-12


In [ ]:


def prepare_market_groups(market_data):
    """Agrupa los arrays requeridos por COS para cada vencimiento."""
    groups = []
    for (expiry, T), expiry_data in market_data.groupby(["expiry", "T"], sort=True):
        rate = float(expiry_data["r"].iloc[0])
        dividend_yield = float(expiry_data["q"].iloc[0])
        spot = float(expiry_data["S0"].iloc[0])
        groups.append({
            "expiry": expiry,
            "T": float(T),
            "r": rate,
            "q": dividend_yield,
            "S0": spot,
            "S_eff": spot * np.exp(-dividend_yield * float(T)),
            "idx": expiry_data.index.to_numpy(),
            "K": expiry_data["K"].to_numpy(dtype=float),
            "type": expiry_data["type"].to_numpy(),
            "iv_mkt": expiry_data["iv_mkt"].to_numpy(dtype=float),
            "vega": expiry_data["vega"].to_numpy(dtype=float),
        })
    return groups


MARKET_GROUPS = prepare_market_groups(calib)
N_OBS = len(calib)
print(f"Cadena calibrable: {N_OBS} observaciones en {len(MARKET_GROUPS)} vencimientos.")
for grp in MARKET_GROUPS:
    print(f"  T={grp['T']:.4f}  n={len(grp['K'])}  S_eff={grp['S_eff']:.2f}")


In [ ]:


COS_N_CAL = 256
COS_L_CAL = 12
BAD_RESIDUAL = 1.0


def bs_price_vectorized(S0, K, T, r, q, sigma, is_call):
    """Precio BS forward vectorizado sobre arrays K, sigma, is_call (bool)."""
    K = np.asarray(K, float)
    sigma = np.asarray(sigma, float)
    F = S0 * np.exp((r - q) * T)
    disc = np.exp(-r * T)
    vol = sigma * np.sqrt(T)
    vol = np.where(vol > 0, vol, np.nan)
    d1 = (np.log(F / K) + 0.5 * vol * vol) / vol
    d2 = d1 - vol
    call = disc * (F * norm.cdf(d1) - K * norm.cdf(d2))
    put = disc * (K * norm.cdf(-d2) - F * norm.cdf(-d1))
    return np.where(is_call, call, put)


def bs_vega_vectorized(S0, K, T, r, q, sigma):
    """Vega BS forward vectorizado (igual para call y put)."""
    K = np.asarray(K, float)
    sigma = np.asarray(sigma, float)
    F = S0 * np.exp((r - q) * T)
    disc = np.exp(-r * T)
    vol = sigma * np.sqrt(T)
    vol = np.where(vol > 0, vol, np.nan)
    d1 = (np.log(F / K) + 0.5 * vol * vol) / vol
    return disc * F * norm.pdf(d1) * np.sqrt(T)


def implied_vol_vectorized(price, S0, K, T, r, q, is_call,
                    lo=1e-4, hi=5.0, tol=1e-8, maxiter=60):
    """IV BS vectorizada por Newton con respaldo de biseccion. NaN fuera de cotas.

    price, K, is_call son arrays; (S0, T, r, q) escalares del vencimiento. Devuelve
    un array de IV alineado con K. Misma convencion forward que implied_vol."""
    price = np.asarray(price, float)
    K = np.asarray(K, float)
    is_call = np.asarray(is_call, bool)
    n = len(K)
    iv = np.full(n, np.nan)

    F = S0 * np.exp((r - q) * T)
    disc = np.exp(-r * T)
    lower = np.where(is_call, disc * np.maximum(F - K, 0.0),
                     disc * np.maximum(K - F, 0.0))
    upper = np.where(is_call, disc * F, disc * K)
    eps = 1e-10
    valid = (np.isfinite(price) & (price > 0) & (T > 0)
             & (price > lower + eps) & (price < upper - eps))
    if not np.any(valid):
        return iv

    idx = np.where(valid)[0]
    p = price[idx]
    Kv = K[idx]
    cv = is_call[idx]
    # Biseccion vectorizada robusta (garantiza convergencia dentro de [lo, hi]).
    a = np.full(len(idx), lo)
    b = np.full(len(idx), hi)
    fa = bs_price_vectorized(S0, Kv, T, r, q, a, cv) - p
    for _ in range(maxiter):
        m = 0.5 * (a + b)
        fm = bs_price_vectorized(S0, Kv, T, r, q, m, cv) - p
        left = np.sign(fm) == np.sign(fa)
        a = np.where(left, m, a)
        fa = np.where(left, fm, fa)
        b = np.where(left, b, m)
        if np.all((b - a) < tol):
            break
    sol = 0.5 * (a + b)
    # Pulido Newton (vega analitica) para precision de maquina donde vega no es nula.
    for _ in range(3):
        f = bs_price_vectorized(S0, Kv, T, r, q, sol, cv) - p
        v = bs_vega_vectorized(S0, Kv, T, r, q, sol)
        step = np.where(np.abs(v) > 1e-12, f / v, 0.0)
        sol_new = np.clip(sol - step, lo, hi)
        sol = sol_new
    iv[idx] = sol
    iv[idx] = np.where(np.isfinite(iv[idx]), iv[idx], np.nan)
    return iv


def model_prices_group(group, sigma0, sigma1, lam0, lam1, N=COS_N_CAL, L=COS_L_CAL):
    """Precio no condicionado C_mod = pi0*C0 + pi1*C1 para un vencimiento.

    Vectorizado sobre los strikes del grupo. COS valora calls y las puts se obtienen
    por paridad, valida porque r0=r1. Devuelve un array alineado con grp['K']."""
    T = group["T"]
    r = group["r"]
    S_eff = group["S_eff"]
    K = group["K"]
    types = group["type"]
    pi0, pi1 = stationary_weights(lam0, lam1)

    # Con r0=r1 la paridad vale por regimen. Valoramos TODAS las calls en dos
    # evaluaciones COS (una por regimen) y obtenemos puts por paridad usando el
    # spot ajustado: P=C-S_eff+K*exp(-rT). Esto reduce a la mitad el costo del
    # objetivo sin cambiar el modelo.
    c0 = cos_method(S_eff, K, T, r, sigma0, sigma1, lam0, lam1,
                    regime=0, N=N, L=L, option_type="call")
    c1 = cos_method(S_eff, K, T, r, sigma0, sigma1, lam0, lam1,
                    regime=1, N=N, L=L, option_type="call")
    out = pi0 * np.asarray(c0) + pi1 * np.asarray(c1)
    put_mask = (types == "put")
    out[put_mask] = out[put_mask] - S_eff + K[put_mask] * np.exp(-r * T)
    return out


def model_iv_group(group, sigma0, sigma1, lam0, lam1, N=COS_N_CAL, L=COS_L_CAL):
    """IV del modelo por vencimiento, invertida con la misma convencion (r, q).

    Devuelve (iv_model, price_model). IV NaN donde el precio sea invalido o no
    invertible (la deteccion de invalidos la gestiona el objetivo)."""
    prices = model_prices_group(group, sigma0, sigma1, lam0, lam1, N=N, L=L)
    T, r, q = group["T"], group["r"], group["q"]
    S0_g = group["S0"]
    K = group["K"]
    is_call = group["type"] == "call"
    # Inversion VECTORIZADA con la misma convencion forward (r, q) que la IV de
    # mercado: Newton + biseccion sobre todos los strikes del vencimiento a la vez.
    iv = implied_vol_vectorized(prices, S0_g, K, T, r, q, is_call)
    return iv, prices


_grp0 = MARKET_GROUPS[0]
_ivm, _pm = model_iv_group(_grp0, 0.15, 0.40, 2.0, 5.0)
_ok = np.isfinite(_ivm)
print(f"Vencimiento T={_grp0['T']:.4f}: IV de modelo finita en {int(_ok.sum())}/{len(_ivm)} strikes.")
print("  IV modelo (primeros 5):", np.round(_ivm[:5], 4))
print("  IV mercado (primeros 5):", np.round(_grp0['iv_mkt'][:5], 4))

_iv_scalar = np.array([
    implied_vol(float(_pm[j]), _grp0["S0"], float(_grp0["K"][j]), _grp0["T"],
                _grp0["r"], _grp0["q"], str(_grp0["type"][j]))
    for j in range(len(_grp0["K"]))
])
_both = np.isfinite(_ivm) & np.isfinite(_iv_scalar)
print("  max|IV vectorizada - IV brentq| :",
      float(np.max(np.abs(_ivm[_both] - _iv_scalar[_both]))))


In [ ]:


def calibration_rmse(u, weighting="uniform", groups=None,
                     N=COS_N_CAL, L=COS_L_CAL, bad=BAD_RESIDUAL,
                     return_detail=False):
    """RMSE ponderado en IV entre modelo y mercado. u en R^4 (transformado).

    weighting: 'uniform' o 'vega'. Devuelve un float finito. Con return_detail=True
    devuelve tambien (residuos, pesos, iv_model) para diagnostico."""
    if groups is None:
        groups = MARKET_GROUPS
    sigma0, sigma1, lam0, lam1 = theta_from_u(u)

    resid_parts = []
    weight_parts = []
    ivmod_parts = []
    try:
        for grp in groups:
            iv_mod, _ = model_iv_group(grp, sigma0, sigma1, lam0, lam1, N=N, L=L)
            iv_mkt = grp["iv_mkt"]
            r = iv_mod - iv_mkt
            # Residuo de respaldo (acotado) donde la IV del modelo no es valida.
            invalid = ~np.isfinite(r)
            r = np.where(invalid, bad, r)
            r = np.clip(r, -bad, bad)  # acota outliers para que el objetivo sea finito
            if weighting == "vega":
                w = np.asarray(grp["vega"], dtype=float)
                w = np.where(np.isfinite(w) & (w > 0), w, 0.0)
            elif weighting == "uniform":
                w = np.ones(len(r), dtype=float)
            else:
                raise ValueError("weighting debe ser 'uniform' o 'vega'.")
            resid_parts.append(r)
            weight_parts.append(w)
            ivmod_parts.append(iv_mod)
    except Exception:
        # Cualquier fallo numerico inesperado -> escalar finito grande.
        return (bad if not return_detail else (bad, None, None, None))

    resid = np.concatenate(resid_parts)
    weights = np.concatenate(weight_parts)
    tot_w = weights.sum()
    if not np.isfinite(tot_w) or tot_w <= 0:
        weights = np.ones_like(weights)
        tot_w = weights.sum()
    weights = weights * (len(weights) / tot_w)
    mse = float(np.sum(weights * resid ** 2) / len(resid))
    rmse = float(np.sqrt(mse))
    if not np.isfinite(rmse):
        rmse = bad
    if return_detail:
        return rmse, resid, weights, np.concatenate(ivmod_parts)
    return rmse


def calibration_rmse_theta(theta, weighting="uniform", **kw):
    s0, s1, l0, l1 = map(float, theta)
    if not (0 < s0 < s1
            and s0 <= SIGMA0_CEIL + 1e-12 and s1 <= SIGMA1_CEIL + 1e-12
            and LAM_FLOOR - 1e-12 <= l0 <= LAM_CEIL + 1e-12
            and LAM_FLOOR - 1e-12 <= l1 <= LAM_CEIL + 1e-12):
        return float(kw.get("bad", BAD_RESIDUAL))
    return calibration_rmse(u_from_theta(*theta), weighting=weighting, **kw)


print("Objetivo de calibracion listo (RMSE IV, ponderaciones 'uniform' y 'vega').")


In [ ]:

import time as _time

_theta_test = (0.15, 0.40, 2.0, 5.0)
_u_test = u_from_theta(*_theta_test)

_rmse_u = calibration_rmse(_u_test, weighting="uniform")
_rmse_v = calibration_rmse(_u_test, weighting="vega")
print("CRITERIO 1 - objetivo finito en theta de prueba", _theta_test)
print(f"  RMSE IV (uniforme) = {_rmse_u:.6f}  finito={np.isfinite(_rmse_u)}")
print(f"  RMSE IV (vega)     = {_rmse_v:.6f}  finito={np.isfinite(_rmse_v)}")

_rmse_extreme = calibration_rmse(np.array([5.0, 5.0, 10.0, -10.0]), weighting="vega")
_rmse_tiny = calibration_rmse(np.array([-30.0, -30.0, -30.0, -30.0]), weighting="uniform")
print(f"  RMSE en u extremo  = {_rmse_extreme:.6f}  finito={np.isfinite(_rmse_extreme)}")
print(f"  RMSE en u diminuto = {_rmse_tiny:.6f}  finito={np.isfinite(_rmse_tiny)}")

print("CRITERIO 2 - dos ponderaciones:",
      f"uniforme={_rmse_u:.6f} vs vega={_rmse_v:.6f} (distintas={abs(_rmse_u - _rmse_v) > 1e-9})")

_rng = np.random.default_rng(SEED)
_feasible = True
for _ in range(2000):
    _u = _rng.normal(0.0, 5.0, size=4)
    _s0, _s1, _l0, _l1 = theta_from_u(_u)
    if not (0.0 < _s0 < _s1 and _l0 > 0.0 and _l1 > 0.0):
        _feasible = False
        break
print("CRITERIO 3 - 2000 u aleatorios -> theta factible (0<s0<s1, lam>0):", _feasible)

_pi0, _pi1 = stationary_weights(*_theta_test[2:])
print(f"CRITERIO 4 - pi0+pi1 = {_pi0 + _pi1:.12f}  (pi0={_pi0:.6f}, pi1={_pi1:.6f})")

_n_rep = 20
_t0 = _time.perf_counter()
for _ in range(_n_rep):
    calibration_rmse(_u_test, weighting="vega")
_dt = (_time.perf_counter() - _t0) / _n_rep
print(f"CRITERIO 5 - tiempo medio por evaluacion = {_dt * 1e3:.2f} ms"
      f"  (~{1.0 / _dt:.0f} evals/s; 1000 evals ~ {1000 * _dt:.1f} s)")


### Optimizadores: malla, Nelder–Mead y BFGS

Se comparan brute force, Nelder-Mead y BFGS para ambos esquemas de pesos. Los
métodos locales parten del mejor nodo de la malla y se prueba un arranque adicional.

In [ ]:
import time as _t6
from scipy.optimize import minimize

np.random.seed(SEED)
random.seed(SEED)

SIGMA0_BOX = (0.05, 0.30)
SIGMA1_BOX = (0.20, 0.90)   # alineado con SIGMA1_CEIL del transform acotado
LAM0_BOX = (0.5, 8.0)
LAM1_BOX = (0.5, 8.0)

N_S0, N_S1, N_L0, N_L1 = 5, 5, 4, 4

WEIGHTINGS = ["uniform", "vega"]
PRIMARY_WEIGHTING = "vega"  # fijada ex ante; no se elige comparando objetivos distintos

print("Caja de busqueda (espacio original theta):")
print(f"  sigma0 in {SIGMA0_BOX}   sigma1 in {SIGMA1_BOX}")
print(f"  lam0   in {LAM0_BOX}   lam1   in {LAM1_BOX}")
print(f"Malla: {N_S0}x{N_S1}x{N_L0}x{N_L1} = {N_S0*N_S1*N_L0*N_L1} nodos por ponderacion.")


In [ ]:

_s0_grid = np.linspace(*SIGMA0_BOX, N_S0)
_s1_grid = np.linspace(*SIGMA1_BOX, N_S1)
_l0_grid = np.linspace(*LAM0_BOX, N_L0)
_l1_grid = np.linspace(*LAM1_BOX, N_L1)


def brute_force_search(weighting):
    """Malla exhaustiva sobre la caja. Devuelve (best_theta, best_rmse, n_eval, secs)."""
    best_theta = None
    best_rmse = np.inf
    n_eval = 0
    t0 = _t6.perf_counter()
    for s0 in _s0_grid:
        for s1 in _s1_grid:
            if not (s0 < s1):
                continue  # impone 0 < sigma0 < sigma1
            for l0 in _l0_grid:
                for l1 in _l1_grid:
                    rmse = calibration_rmse_theta((s0, s1, l0, l1), weighting=weighting)
                    n_eval += 1
                    if rmse < best_rmse:
                        best_rmse = rmse
                        best_theta = (float(s0), float(s1), float(l0), float(l1))
    secs = _t6.perf_counter() - t0
    return best_theta, float(best_rmse), n_eval, secs


brute_results = {}
for w in WEIGHTINGS:
    bt, br, ne, sc = brute_force_search(w)
    brute_results[w] = {"theta": bt, "rmse": br, "n_eval": ne, "time": sc}
    print(f"[brute force | pesos={w:7s}] "
          f"theta=({bt[0]:.4f},{bt[1]:.4f},{bt[2]:.3f},{bt[3]:.3f})  "
          f"RMSE={br:.6f}  evals={ne}  t={sc:.2f}s")


In [ ]:

def run_local(method, u0, weighting):
    """Corre un optimizador local. method in {'Nelder-Mead','BFGS'}."""
    nfev = {"n": 0}

    def obj(u):
        nfev["n"] += 1
        return calibration_rmse(u, weighting=weighting)

    if method == "Nelder-Mead":
        opts = {"xatol": 1e-6, "fatol": 1e-8, "maxiter": 2000, "maxfev": 2000}
    else:  # BFGS (gradiente por diferencias finitas de scipy)
        opts = {"gtol": 1e-6, "maxiter": 500}

    t0 = _t6.perf_counter()
    res = minimize(obj, np.asarray(u0, float), method=method, options=opts)
    secs = _t6.perf_counter() - t0
    theta = theta_from_u(res.x)
    return {
        "method": method,
        "weighting": weighting,
        "theta": theta,
        "rmse": float(res.fun),
        "n_eval": int(nfev["n"]),
        "time": float(secs),
        "success": bool(res.success),
        "status": str(res.message),
        "x0_theta": theta_from_u(u0),
    }


local_results = []
for w in WEIGHTINGS:
    u0_grid = u_from_theta(*brute_results[w]["theta"])
    for method in ("Nelder-Mead", "BFGS"):
        r = run_local(method, u0_grid, w)
        local_results.append(r)
        th = r["theta"]
        print(f"[{method:11s} | pesos={w:7s}] "
              f"theta=({th[0]:.4f},{th[1]:.4f},{th[2]:.3f},{th[3]:.3f})  "
              f"RMSE={r['rmse']:.6f}  evals={r['n_eval']}  t={r['time']:.2f}s  "
              f"ok={r['success']}")


In [ ]:

EXTRA_START = (0.12, 0.35, 2.0, 4.0)

sens_rows = []
for r0 in local_results:
    r = dict(r0)
    r["start_name"] = "malla"
    sens_rows.append(r)

u0_extra = u_from_theta(*EXTRA_START)
for method in ("Nelder-Mead", "BFGS"):
    r = run_local(method, u0_extra, PRIMARY_WEIGHTING)
    r["start_name"] = "razonable"
    sens_rows.append(r)

print("Sensibilidad al punto inicial (malla + arranques adicionales):")
for w in WEIGHTINGS:
    for method in ("Nelder-Mead", "BFGS"):
        sub = [x for x in sens_rows if x["weighting"] == w and x["method"] == method]
        rmses = np.array([x["rmse"] for x in sub])
        thetas = np.array([x["theta"] for x in sub])
        rmse_spread = float(rmses.max() - rmses.min())
        theta_spread = np.max(np.abs(thetas - thetas.mean(axis=0)), axis=0)
        print(f"  [{method:11s} | {w:7s} | n_starts={len(sub)}] "
              f"RMSE in [{rmses.min():.6f}, {rmses.max():.6f}] "
              f"(spread={rmse_spread:.2e})  max|theta-mean|={np.round(theta_spread,4)}")


In [ ]:
import pandas as pd

rows = []
for w in WEIGHTINGS:
    b = brute_results[w]
    rows.append({
        "Metodo": "brute force", "Pesos": w, "Punto inicial": "(malla)",
        "sigma0": b["theta"][0], "sigma1": b["theta"][1],
        "lam0": b["theta"][2], "lam1": b["theta"][3],
        "RMSE_IV": b["rmse"], "Evals": b["n_eval"], "Tiempo_s": b["time"],
        "Estado": "malla completa",
    })
for r in local_results:
    x0 = r["x0_theta"]
    rows.append({
        "Metodo": r["method"], "Pesos": r["weighting"],
        "Punto inicial": f"({x0[0]:.3f},{x0[1]:.3f},{x0[2]:.2f},{x0[3]:.2f})",
        "sigma0": r["theta"][0], "sigma1": r["theta"][1],
        "lam0": r["theta"][2], "lam1": r["theta"][3],
        "RMSE_IV": r["rmse"], "Evals": r["n_eval"], "Tiempo_s": r["time"],
        "Estado": ("convergio" if r["success"] else "no convergio"),
    })

calib_table = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.6f}")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print("=== Tabla comparativa de calibraciones (Fase 6) ===")
print(calib_table.to_string(index=False))

OUTPUT.mkdir(parents=True, exist_ok=True)
_tbl_path = OUTPUT / "f6_tabla_calibracion.csv"
calib_table.to_csv(_tbl_path, index=False)
print(f"\nTabla guardada en {_tbl_path}")


In [ ]:

def is_feasible(theta, tol=1e-9, enforce_calibration_bounds=True):
    s0, s1, l0, l1 = theta
    basic = (s0 > 0) and (s1 > s0 + tol) and (l0 > 0) and (l1 > 0)
    if not basic:
        return False
    if enforce_calibration_bounds:
        return ((s0 <= SIGMA0_CEIL + tol) and (s1 <= SIGMA1_CEIL + tol)
                and (LAM_FLOOR - tol <= l0 <= LAM_CEIL + tol)
                and (LAM_FLOOR - tol <= l1 <= LAM_CEIL + tol))
    return True


# La ponderacion primaria se fija ANTES de mirar los minimos. No es valido elegir
# entre 'uniform' y 'vega' comparando directamente sus RMSE: son objetivos distintos.
# Elegimos vega porque privilegia cotizaciones con mayor sensibilidad informativa.
candidates = [r for r in local_results
              if r["weighting"] == PRIMARY_WEIGHTING
              and is_feasible(r["theta"]) and np.isfinite(r["rmse"])]
converged_candidates = [r for r in candidates if r["success"]]
if converged_candidates:
    candidates = converged_candidates
candidates_sorted = sorted(candidates, key=lambda r: r["rmse"])

best = candidates_sorted[0]
near = [x for x in sens_rows
        if x["weighting"] == PRIMARY_WEIGHTING
        and abs(x["rmse"] - best["rmse"]) < 5e-4 and is_feasible(x["theta"])]
print(f"Candidatos factibles con ponderacion primaria '{PRIMARY_WEIGHTING}' ordenados por RMSE:")
for r in candidates_sorted:
    th = r["theta"]
    print(f"  {r['method']:11s} | {r['weighting']:7s}  RMSE={r['rmse']:.6f}  "
          f"theta=({th[0]:.4f},{th[1]:.4f},{th[2]:.3f},{th[3]:.3f})")

FINAL = best
FINAL_THETA = FINAL["theta"]
print("\n=== SOLUCION FINAL SELECCIONADA ===")
print(f"  Metodo={FINAL['method']}  Pesos={FINAL['weighting']}")
print(f"  sigma0={FINAL_THETA[0]:.6f}  sigma1={FINAL_THETA[1]:.6f}  "
      f"lam0={FINAL_THETA[2]:.6f}  lam1={FINAL_THETA[3]:.6f}")
print(f"  RMSE IV = {FINAL['rmse']:.6f}")
print(f"  Factible (0<s0<s1, lam>0): {is_feasible(FINAL_THETA)}")
print(f"  Estabilidad: {len(near)} corridas distintas dentro de 5e-4 del mejor RMSE")
pi0_f, pi1_f = stationary_weights(FINAL_THETA[2], FINAL_THETA[3])
print(f"  Mezcla estacionaria: pi0={pi0_f:.4f}, pi1={pi1_f:.4f} (suma={pi0_f+pi1_f:.6f})")


In [ ]:

s0_f, s1_f, l0_f, l1_f = FINAL_THETA
pi0_f, pi1_f = stationary_weights(l0_f, l1_f)

_calib_sorted = calib.sort_values(["T", "moneyness"]).reset_index(drop=True)
_pick_idx = np.linspace(0, len(_calib_sorted) - 1, 4).round().astype(int)
verif_rows = []
for j in _pick_idx:
    obs = _calib_sorted.iloc[int(j)]
    T = float(obs["T"]); r = float(obs["r"]); q = float(obs["q"])
    K = float(obs["K"]); otype = str(obs["type"]); S0_o = float(obs["S0"])
    S_eff = S0_o * np.exp(-q * T)

    c0_cos = float(cos_method(S_eff, np.array([K]), T, r, s0_f, s1_f, l0_f, l1_f,
                              regime=0, N=COS_N, L=COS_L, option_type=otype)[0])
    c1_cos = float(cos_method(S_eff, np.array([K]), T, r, s0_f, s1_f, l0_f, l1_f,
                              regime=1, N=COS_N, L=COS_L, option_type=otype)[0])
    price_cos = pi0_f * c0_cos + pi1_f * c1_cos

    c0_a = price_analytic_markov(S_eff, K, T, r, r, s0_f, s1_f, l0_f, l1_f,
                                 regime=0, option_type=otype)
    c1_a = price_analytic_markov(S_eff, K, T, r, r, s0_f, s1_f, l0_f, l1_f,
                                 regime=1, option_type=otype)
    price_A = pi0_f * c0_a + pi1_f * c1_a

    verif_rows.append({
        "T": T, "tipo": otype, "K": K, "moneyness": float(obs["moneyness"]),
        "precio_COS": price_cos, "precio_RutaA": price_A,
        "abs_diff": abs(price_cos - price_A),
        "rel_diff": abs(price_cos - price_A) / max(price_A, 1e-12),
    })

verif_df = pd.DataFrame(verif_rows)
print("=== Verificacion de precios finales: Ruta B (COS) vs Ruta A (analitica) ===")
print(verif_df.to_string(index=False,
      formatters={"abs_diff": lambda v: f"{v:.2e}", "rel_diff": lambda v: f"{v:.2e}"}))
print(f"\nMax |COS - RutaA| = {verif_df['abs_diff'].max():.3e}")
print(f"Max diff relativa = {verif_df['rel_diff'].max():.3e}")


### Diagnóstico por combinaciones convexas e identificabilidad

Se estudian dos cortes afines del RMSE y perfiles 1D alrededor del óptimo. Los
puntos que violan las restricciones se excluyen.

In [ ]:


ALPHAS = np.linspace(-0.5, 1.5, 81)  # incluye alpha=0 y alpha=1 exactos


def rmse_along_segment(theta1, theta2, weighting="uniform", alphas=ALPHAS):
    """RMSE(IV) sobre theta(alpha)=alpha*theta1+(1-alpha)*theta2.

    Devuelve (alphas, rmse, feasible_mask). alpha=1 -> theta1, alpha=0 -> theta2.
    RMSE es NaN donde la combinacion afin es infactible (0<s0<s1, lam>0 violado),
    de modo que la curva muestra explicitamente la region no factible."""
    theta1 = np.asarray(theta1, dtype=float)
    theta2 = np.asarray(theta2, dtype=float)
    alphas = np.asarray(alphas, dtype=float)
    rmse = np.full(len(alphas), np.nan)
    feasible = np.zeros(len(alphas), dtype=bool)
    for k, a in enumerate(alphas):
        th = a * theta1 + (1.0 - a) * theta2
        if is_feasible(tuple(th)):
            feasible[k] = True
            rmse[k] = calibration_rmse_theta(tuple(th), weighting=weighting)
    return alphas, rmse, feasible


DIAG_W = FINAL["weighting"]

theta_mesh = np.array(brute_results[DIAG_W]["theta"], dtype=float)
_bfgs = [r for r in local_results if r["method"] == "BFGS" and r["weighting"] == DIAG_W][0]
_nm = [r for r in local_results if r["method"] == "Nelder-Mead" and r["weighting"] == DIAG_W][0]
theta_bfgs = np.array(_bfgs["theta"], dtype=float)
theta_nm = np.array(_nm["theta"], dtype=float)

print(f"Ponderacion del diagnostico: {DIAG_W}")
print(f"  malla : ({theta_mesh[0]:.4f},{theta_mesh[1]:.4f},{theta_mesh[2]:.3f},{theta_mesh[3]:.3f})  RMSE={brute_results[DIAG_W]['rmse']:.6f}")
print(f"  N-M   : ({theta_nm[0]:.4f},{theta_nm[1]:.4f},{theta_nm[2]:.3f},{theta_nm[3]:.3f})  RMSE={_nm['rmse']:.6f}")
print(f"  BFGS  : ({theta_bfgs[0]:.4f},{theta_bfgs[1]:.4f},{theta_bfgs[2]:.3f},{theta_bfgs[3]:.3f})  RMSE={_bfgs['rmse']:.6f}")

SEGMENTS = [
    ("malla vs BFGS", theta_mesh, theta_bfgs),
    ("Nelder-Mead vs BFGS", theta_nm, theta_bfgs),
]

segment_curves = []
for label, th1, th2 in SEGMENTS:
    a, rm, feas = rmse_along_segment(th1, th2, weighting=DIAG_W)
    segment_curves.append((label, a, rm, feas, th1, th2))
    n_feas = int(feas.sum())
    n_infeas = int((~feas).sum())
    i0 = int(np.argmin(np.abs(a - 0.0)))
    i1 = int(np.argmin(np.abs(a - 1.0)))
    rm_feas = rm[feas]
    print(f"\n[{label}] alpha en [-0.5,1.5]: {n_feas} factibles, {n_infeas} infactibles (recortados)")
    print(f"  RMSE(alpha=0)={rm[i0]:.6f}  RMSE(alpha=1)={rm[i1]:.6f}")
    if rm_feas.size:
        kmin = np.nanargmin(rm)
        print(f"  min RMSE en el segmento = {np.nanmin(rm):.6f} en alpha={a[kmin]:.3f}")
        # Joroba interior: hay un punto factible interior con RMSE > max(extremos)?
        interior = feas & (a > 0.0) & (a < 1.0)
        if np.any(interior):
            hump = np.nanmax(rm[interior]) - max(rm[i0], rm[i1])
            print(f"  joroba interior (max interior - max extremos) = {hump:+.6f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)
for ax, (label, a, rm, feas, th1, th2) in zip(axes, segment_curves):
    ax.plot(a[feas], rm[feas], "-", color="tab:blue", lw=1.8, label="RMSE (factible)")
    ax.plot(a[feas], rm[feas], ".", color="tab:blue", ms=4)
    if np.any(~feas):
        in_a = a[~feas]
        ax.scatter(in_a, np.full(in_a.shape, np.nanmin(rm[feas])),
                   marker="x", color="tab:red", s=20, label="infactible (recortado)")
        for xv in in_a:
            ax.axvspan(xv - 0.0125, xv + 0.0125, color="tab:red", alpha=0.05)
    i0 = int(np.argmin(np.abs(a - 0.0)))
    i1 = int(np.argmin(np.abs(a - 1.0)))
    ax.axvline(0.0, color="gray", ls="--", lw=1)
    ax.axvline(1.0, color="gray", ls="--", lw=1)
    ax.plot(0.0, rm[i0], "o", color="black", ms=8, zorder=5)
    ax.plot(1.0, rm[i1], "s", color="darkgreen", ms=8, zorder=5)
    ax.annotate(r"$\alpha=0$" + f"\n{label.split(' vs ')[1]}", (0.0, rm[i0]),
                textcoords="offset points", xytext=(6, 8), fontsize=8)
    ax.annotate(r"$\alpha=1$" + f"\n{label.split(' vs ')[0]}", (1.0, rm[i1]),
                textcoords="offset points", xytext=(6, 8), fontsize=8)
    ax.set_title(f"Corte afin: {label}")
    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel(f"RMSE IV (pesos={DIAG_W})")
    ax.legend(fontsize=8, loc="best")
    ax.grid(alpha=0.3)

fig.suptitle(r"RMSE(IV) a lo largo de $\theta(\alpha)=\alpha\,\hat\theta_1+(1-\alpha)\,\hat\theta_2$",
             fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.96))
_seg_path = OUTPUT / "f7_rmse_segmentos.png"
fig.savefig(_seg_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Figura guardada en {_seg_path}")
print(f"Curvas RMSE vs alpha generadas: {len(segment_curves)}")


In [ ]:


theta_opt = np.array(FINAL_THETA, dtype=float)
PARAM_NAMES = [r"$\sigma_0$", r"$\sigma_1$", r"$\lambda_0$", r"$\lambda_1$"]
PARAM_KEYS = ["sigma0", "sigma1", "lam0", "lam1"]

def profile_param(j, frac=0.30, npts=41):
    base = theta_opt.copy()
    center = base[j]
    lo, hi = center * (1 - frac), center * (1 + frac)
    grid = np.linspace(lo, hi, npts)
    vals = np.full(npts, np.nan)
    for k, v in enumerate(grid):
        th = base.copy()
        th[j] = v
        if is_feasible(tuple(th)):
            vals[k] = calibration_rmse_theta(tuple(th), weighting=DIAG_W)
    return grid, vals


profiles = [profile_param(j) for j in range(4)]

print("Perfiles 1D calculados; los tramos NaN corresponden a parametros infactibles.")


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for j, ax in enumerate(axes.ravel()):
    grid, vals = profiles[j]
    ax.plot(grid, vals, "-o", ms=3, color="tab:blue")
    ax.axvline(theta_opt[j], color="darkgreen", ls="--", lw=1.2,
               label="optimo")
    ax.set_title(f"Perfil 1D: {PARAM_NAMES[j]}")
    ax.set_xlabel(PARAM_NAMES[j])
    ax.set_ylabel(f"RMSE IV ({DIAG_W})")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Perfiles 1D del RMSE alrededor del optimo final", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.96))
_prof_path = OUTPUT / "f7_perfiles_1d.png"
fig.savefig(_prof_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Figura guardada en {_prof_path}")


Una curva suave sin jorobas indica una cuenca común; jorobas o mínimos separados
sugieren óptimos locales. Los perfiles planos señalan parámetros débiles y una cota
activa debe interpretarse como falta de identificación, no como una estimación precisa.

## Parte D — Análisis e interpretación

### Sonrisa de IV y residuos

Se comparan sonrisa, residuos, Black-Scholes plano, calibraciones por plazo y
magnitudes económicas. Todas las salidas se guardan en `output/`.

In [ ]:
s0_F, s1_F, l0_F, l1_F = FINAL_THETA
pi0_F, pi1_F = stationary_weights(l0_F, l1_F)

rows_iv = []
for grp in MARKET_GROUPS:
    iv_mod, price_mod = model_iv_group(grp, s0_F, s1_F, l0_F, l1_F)
    for j, gi in enumerate(grp["idx"]):
        rows_iv.append({
            "idx": int(gi),
            "expiry": grp["expiry"],
            "T": grp["T"],
            "K": float(grp["K"][j]),
            "type": str(grp["type"][j]),
            "moneyness": float(grp["K"][j] / grp["S0"]),
            "iv_mkt": float(grp["iv_mkt"][j]),
            "iv_mod": float(iv_mod[j]),
            "vega": float(grp["vega"][j]),
        })

iv_df = pd.DataFrame(rows_iv).set_index("idx").sort_values(["T", "moneyness"])
iv_df["resid"] = iv_df["iv_mod"] - iv_df["iv_mkt"]
n_fin = int(np.isfinite(iv_df["iv_mod"]).sum())
print(f"IV de modelo finita en {n_fin}/{len(iv_df)} observaciones.")
rmse_reg_global = float(np.sqrt(np.nanmean(iv_df["resid"].to_numpy() ** 2)))
print(f"RMSE IV (modelo de regimenes, no ponderado) sobre la cadena = {rmse_reg_global:.6f}")
print(f"theta final: sigma0={s0_F:.4f}, sigma1={s1_F:.4f}, lam0={l0_F:.4f}, lam1={l1_F:.4f}")
print(f"pi0={pi0_F:.4f}, pi1={pi1_F:.4f}")

In [ ]:
expiries = list(iv_df["T"].drop_duplicates())
n_exp = len(expiries)
ncol = 3
nrow = int(np.ceil(n_exp / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 3.4 * nrow), squeeze=False)
for ax in axes.flat:
    ax.set_visible(False)
for i, T in enumerate(expiries):
    ax = axes.flat[i]
    ax.set_visible(True)
    sub = iv_df[iv_df["T"] == T].sort_values("moneyness")
    fin = sub[np.isfinite(sub["iv_mod"])]
    ax.scatter(sub["moneyness"], sub["iv_mkt"], s=14, alpha=0.55,
               color="tab:blue", label="Mercado")
    ax.plot(fin["moneyness"], fin["iv_mod"], "-", color="tab:red", lw=1.6,
            label="Modelo regimenes")
    ax.set_title(f"T = {T:.3f} anos  (n={len(sub)})")
    ax.set_xlabel("Moneyness K/S0")
    ax.set_ylabel("IV")
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8)
fig.suptitle("Sonrisa de volatilidad implicita: mercado vs modelo de regimenes",
             y=1.002, fontsize=13)
fig.tight_layout()
FIG_SMILE = OUTPUT / "f8_sonrisa_iv_mercado_vs_modelo.png"
fig.savefig(FIG_SMILE, dpi=130, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {FIG_SMILE}")

In [ ]:
res = iv_df[np.isfinite(iv_df["resid"])].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

sc = axes[0].scatter(res["moneyness"], res["resid"], c=res["T"], cmap="viridis",
                     s=16, alpha=0.7)
axes[0].axhline(0.0, color="k", lw=0.8)
axes[0].set_xlabel("Moneyness K/S0")
axes[0].set_ylabel("Residuo IV (modelo - mercado)")
axes[0].set_title("Residuos por moneyness")
axes[0].grid(alpha=0.3)
cb = fig.colorbar(sc, ax=axes[0]); cb.set_label("T (anos)")

Ts = sorted(res["T"].unique())
data = [res[res["T"] == T]["resid"].to_numpy() for T in Ts]
axes[1].boxplot(data, tick_labels=[f"{T:.2f}" for T in Ts], showmeans=True)
axes[1].axhline(0.0, color="k", lw=0.8)
axes[1].set_xlabel("Vencimiento T (anos)")
axes[1].set_ylabel("Residuo IV")
axes[1].set_title("Distribucion de residuos por vencimiento")
axes[1].grid(alpha=0.3)
fig.tight_layout()
FIG_RESID = OUTPUT / "f8_residuos_iv.png"
fig.savefig(FIG_RESID, dpi=130, bbox_inches="tight")
plt.show()

resumen_resid = (res.groupby("T")
                 .agg(n=("resid", "size"),
                      resid_medio=("resid", "mean"),
                      resid_abs_medio=("resid", lambda x: np.mean(np.abs(x))),
                      rmse=("resid", lambda x: np.sqrt(np.mean(x ** 2))))
                 .reset_index())
print("Residuos de IV por vencimiento (modelo de regimenes):")
print(resumen_resid.to_string(index=False,
      formatters={c: (lambda v: f"{v:.4f}") for c in
                  ["resid_medio", "resid_abs_medio", "rmse"]}))
print(f"\nFigura guardada: {FIG_RESID}")

### Black-Scholes plano

Se calibra un único $\sigma$ sobre el mismo conjunto, criterio y ponderaciones.

In [ ]:
from scipy.optimize import minimize_scalar

iv_mkt_all = calib["iv_mkt"].to_numpy(dtype=float)
vega_all = calib["vega"].to_numpy(dtype=float)
mask_iv = np.isfinite(iv_mkt_all)

def bs_flat_rmse(sigma, weighting="uniform"):
    r = sigma - iv_mkt_all[mask_iv]
    if weighting == "vega":
        w = vega_all[mask_iv]
        w = np.where(np.isfinite(w) & (w > 0), w, 0.0)
    else:
        w = np.ones_like(r)
    tot = w.sum()
    if not np.isfinite(tot) or tot <= 0:
        w = np.ones_like(r); tot = w.sum()
    w = w * (len(w) / tot)  # misma normalizacion (media 1) que el objetivo del modelo
    return float(np.sqrt(np.sum(w * r ** 2) / len(r)))

bs_flat = {}
for w in ("uniform", "vega"):
    opt = minimize_scalar(lambda s: bs_flat_rmse(s, w), bounds=(0.01, 2.0),
                          method="bounded", options={"xatol": 1e-8})
    bs_flat[w] = {"sigma": float(opt.x), "rmse": float(opt.fun)}
    print(f"BS plano [{w:7s}]: sigma* = {opt.x:.6f}   RMSE IV = {opt.fun:.6f}")

reg_rmse = {w: float(calibration_rmse_theta(FINAL_THETA, weighting=w))
            for w in ("uniform", "vega")}
print()
for w in ("uniform", "vega"):
    print(f"Modelo regimenes [{w:7s}]: RMSE IV = {reg_rmse[w]:.6f}")

w_final = FINAL["weighting"]
rmse_bs_f = bs_flat[w_final]["rmse"]
rmse_reg_f = reg_rmse[w_final]
mejora_abs = rmse_bs_f - rmse_reg_f
mejora_rel = mejora_abs / rmse_bs_f * 100.0
print()
print("=== Comparacion BS plano vs modelo de regimenes (esquema de la solucion final:"
      f" {w_final}) ===")
print(f"  RMSE BS plano        = {rmse_bs_f:.6f}  (1 parametro)")
print(f"  RMSE modelo regimenes= {rmse_reg_f:.6f}  (4 parametros)")
print(f"  Mejora absoluta en RMSE IV = {mejora_abs:.6f}")
print(f"  Mejora relativa            = {mejora_rel:.2f}%")
comp_bs_df = pd.DataFrame([
    {"modelo": "BS plano", "n_param": 1, "weighting": w,
     "sigma_o_theta": f"sigma={bs_flat[w]['sigma']:.4f}", "rmse_iv": bs_flat[w]["rmse"]}
    for w in ("uniform", "vega")
] + [
    {"modelo": "Regimenes", "n_param": 4, "weighting": w,
     "sigma_o_theta": f"s0={s0_F:.3f},s1={s1_F:.3f},l0={l0_F:.3f},l1={l1_F:.3f}",
     "rmse_iv": reg_rmse[w]}
    for w in ("uniform", "vega")
])
comp_bs_df.to_csv(OUTPUT / "f8_comparacion_bs_vs_regimenes.csv", index=False)
print("\nTabla guardada: " + str(OUTPUT / "f8_comparacion_bs_vs_regimenes.csv"))

### Calibración por vencimiento

Cada plazo se recalibra desde la solución conjunta para evaluar estabilidad temporal.

In [ ]:
from scipy.optimize import minimize as _minimize_f8

u0_joint = u_from_theta(*FINAL_THETA)
w_cal = FINAL["weighting"]

per_expiry = []
for grp in MARKET_GROUPS:
    def obj_T(u, g=grp):
        return calibration_rmse(u, weighting=w_cal, groups=[g])
    res = _minimize_f8(obj_T, u0_joint, method="Nelder-Mead",
                       options={"xatol": 1e-6, "fatol": 1e-8,
                                "maxiter": 2000, "maxfev": 2000})
    th = theta_from_u(res.x)
    at_boundary = (th[0] >= 0.995 * SIGMA0_CEIL
                   or th[1] >= 0.995 * SIGMA1_CEIL
                   or th[2] <= 1.005 * LAM_FLOOR or th[2] >= 0.995 * LAM_CEIL
                   or th[3] <= 1.005 * LAM_FLOOR or th[3] >= 0.995 * LAM_CEIL)
    per_expiry.append({
        "T": grp["T"], "n": len(grp["K"]),
        "sigma0": th[0], "sigma1": th[1], "lam0": th[2], "lam1": th[3],
        "rmse": float(res.fun), "success": bool(res.success),
        "boundary": bool(at_boundary),
    })

plazo_df = pd.DataFrame(per_expiry).sort_values("T").reset_index(drop=True)
print("=== Calibracion POR VENCIMIENTO (ponderacion: " + w_cal + ") ===")
print(plazo_df.to_string(index=False,
      formatters={c: (lambda v: f"{v:.4f}") for c in
                  ["sigma0", "sigma1", "lam0", "lam1", "rmse"]}))
print(f"\nVencimientos con alguna cota activa: {int(plazo_df['boundary'].sum())}/{len(plazo_df)}")

print("\n=== Calibracion CONJUNTA (FINAL_THETA) ===")
print(f"  sigma0={s0_F:.4f}  sigma1={s1_F:.4f}  lam0={l0_F:.4f}  lam1={l1_F:.4f}")

print("\nEstabilidad a traves del plazo (media +/- desv, coef. de variacion):")
for p in ["sigma0", "sigma1", "lam0", "lam1"]:
    vals = plazo_df[p].to_numpy()
    cv = float(np.std(vals) / np.abs(np.mean(vals))) if np.mean(vals) != 0 else np.nan
    print(f"  {p:7s}: media={np.mean(vals):.4f}  desv={np.std(vals):.4f}  CV={cv:.2f}")
plazo_df.to_csv(OUTPUT / "f8_calibracion_por_vencimiento.csv", index=False)
print("\nTabla guardada: " + str(OUTPUT / "f8_calibracion_por_vencimiento.csv"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
T_arr = plazo_df["T"].to_numpy()

axes[0].plot(T_arr, plazo_df["sigma0"], "o-", label=r"$\sigma_0$ separado", color="tab:blue")
axes[0].plot(T_arr, plazo_df["sigma1"], "s-", label=r"$\sigma_1$ separado", color="tab:red")
axes[0].axhline(s0_F, ls="--", color="tab:blue", alpha=0.7, label=r"$\sigma_0$ conjunto")
axes[0].axhline(s1_F, ls="--", color="tab:red", alpha=0.7, label=r"$\sigma_1$ conjunto")
axes[0].set_xlabel("T (anos)"); axes[0].set_ylabel("Volatilidad")
axes[0].set_title("Niveles de volatilidad por plazo"); axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].plot(T_arr, plazo_df["lam0"], "o-", label=r"$\lambda_0$ separado", color="tab:green")
axes[1].plot(T_arr, plazo_df["lam1"], "s-", label=r"$\lambda_1$ separado", color="tab:purple")
axes[1].axhline(l0_F, ls="--", color="tab:green", alpha=0.7, label=r"$\lambda_0$ conjunto")
axes[1].axhline(l1_F, ls="--", color="tab:purple", alpha=0.7, label=r"$\lambda_1$ conjunto")
axes[1].set_xlabel("T (anos)"); axes[1].set_ylabel("Intensidad")
axes[1].set_title("Intensidades de cambio de regimen por plazo"); axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

fig.suptitle("Estabilidad a plazo: calibracion separada (puntos) vs conjunta (lineas)",
             y=1.02, fontsize=12)
fig.tight_layout()
FIG_PLAZO = OUTPUT / "f8_estabilidad_plazo.png"
fig.savefig(FIG_PLAZO, dpi=130, bbox_inches="tight")
plt.show()
print(f"Figura guardada: {FIG_PLAZO}")

In [ ]:
dur0 = 1.0 / l0_F
dur1 = 1.0 / l1_F
pi0_F, pi1_F = stationary_weights(l0_F, l1_F)
t_mix = 1.0 / (l0_F + l1_F)

print("=== Interpretacion economica de theta calibrado ===")
print(f"  Regimen 0 (baja vol): sigma0 = {s0_F:.4f}  ({s0_F*100:.1f}% anual)")
print(f"  Regimen 1 (alta vol): sigma1 = {s1_F:.4f}  ({s1_F*100:.1f}% anual)")
print()
print(f"  Duracion esperada en regimen 0: 1/lam0 = {dur0:.4f} anos  (~{dur0*252:.0f} dias habiles)")
print(f"  Duracion esperada en regimen 1: 1/lam1 = {dur1:.4f} anos  (~{dur1*252:.0f} dias habiles)")
print()
print(f"  Probabilidades estacionarias: pi0 = {pi0_F:.4f}, pi1 = {pi1_F:.4f}  (suma={pi0_F+pi1_F:.6f})")
print()
print(f"  Segundo autovalor de Q = -(lam0+lam1) = {-(l0_F+l1_F):.4f}")
print(f"  Tiempo de mezcla ~ 1/(lam0+lam1) = {t_mix:.4f} anos  (~{t_mix*252:.0f} dias habiles)")
print(f"  A T >~ 3*t_mix = {3*t_mix:.3f} anos el peso del regimen inicial decae ~e^-3 (~5%).")

econ_df = pd.DataFrame([{
    "sigma0": s0_F, "sigma1": s1_F, "lam0": l0_F, "lam1": l1_F,
    "dur0_anos": dur0, "dur1_anos": dur1, "pi0": pi0_F, "pi1": pi1_F,
    "tiempo_mezcla_anos": t_mix,
}])
econ_df.to_csv(OUTPUT / "f8_interpretacion_economica.csv", index=False)
print("\nTabla guardada: " + str(OUTPUT / "f8_interpretacion_economica.csv"))

### Respuestas a las seis preguntas de la Parte D

**1. Identificabilidad.** $\sigma_0$ está mejor determinada porque mueve el nivel
principal de la sonrisa. Las intensidades son débiles: los precios identifican mejor
el cociente estacionario que $\lambda_0$ y $\lambda_1$ por separado. Si $\sigma_1$
alcanza su cota, se reporta como dirección no identificada por arriba.

**2. Tasa común.** Con $r_0=r_1=r$, el descuento es $e^{-rT}$ y las tres rutas
son comparables. Si las tasas difieren, el descuento depende del tiempo de ocupación y
COS deja de aplicar. Una sola curva de opciones no separa dos tasas ocultas; harían falta
instrumentos de renta fija o de tasas sensibles al régimen.

**3. Comparación con Black-Scholes.** El modelo plano produce IV constante y no
captura el skew. El modelo de regímenes reduce el RMSE al combinar dos niveles de
volatilidad. La mejora debe evaluarse junto con la identificación débil de las
intensidades y la estructura a plazo residual.

**4. Estructura a plazo.** Las calibraciones separadas muestran si los parámetros
son estables o compensan cambios del skew. Un parámetro en la frontera no está bien
identificado aunque varíe poco. La calibración conjunta usa más información, pero sigue
siendo un compromiso de parámetros constantes.

**5. Interpretación económica.** $\sigma_0$ representa baja volatilidad y
$\sigma_1$ estrés; $1/\lambda_i$ es la duración media y $\pi_i$ la frecuencia de
largo plazo. Si $\sigma_1$ toca el techo, se interpreta como cota activa y no como
un nivel de crisis estimado con precisión.

**6. Horizonte de mezcla.** El efecto del régimen inicial decae como
$e^{-(\lambda_0+\lambda_1)T}$. Por tanto,
$t_{mix}=1/(\lambda_0+\lambda_1)$ y cerca de $3t_{mix}$ queda alrededor de 5% del
efecto inicial. A plazos largos domina la mezcla estacionaria.